In [1]:
%run ../scripts/notebook_settings_lean.py
from scipy import stats
from horizonplot import horizonplot
from chromwindow import window
import zarr
import allel
pd.options.display.float_format = '{:10,.3g}'.format #

A repeat of rfmix14, now with an addendum wherein I compare the Tanzania results with the close references in Serengeti/Mikumi.

In [2]:
c_r_g_df = pd.read_csv("../steps/rfmix_stats_df/call_recomb_genes.txt")
c_r_g_df

,chrom,start,callable_frac,end,cM,end_cM,average_cM_window,genes,genic
0,chr1,0,0.795,100000,0,0.36,3.6e-06,"['RNF223', 'C1H1orf159', 'C1H1orf159', 'C1H1or...",True
1,chr1,100000,0.89,200000,0.36,0.477,1.16e-06,"['TTLL10', 'TTLL10', 'TTLL10', 'TTLL10', 'TTLL...",True
2,chr1,200000,0.919,300000,0.477,0.547,7.05e-07,"['B3GALT6', 'C1QTNF12', 'UBE2J2', 'UBE2J2', 'U...",True
3,chr1,300000,0.878,400000,0.547,0.626,7.94e-07,"['DVL1', 'DVL1', 'DVL1', 'MXRA8', 'MXRA8', 'MX...",True
4,chr1,400000,0.485,500000,0.626,1.37,7.4e-06,"['VWA1', 'TMEM240', 'SSU72']",True
...,...,...,...,...,...,...,...,...,...
27377,chrX,143200000,0.67,143300000,127,127,2.56e-07,"['TMLHE', 'TMLHE', 'TMLHE', 'TMLHE', 'TMLHE']",True
27378,chrX,143300000,0.905,143400000,127,127,3.37e-07,"['TMLHE', 'TMLHE', 'TMLHE', 'TMLHE']",True
27379,chrX,143400000,0.789,143500000,127,127,2.8e-07,['SPRY3'],True
27380,chrX,143500000,0.836,143600000,127,127,7.53e-07,['VAMP7'],True


Section on outgroup diversity

In [3]:
zarr_chrX_dir = "/home/eriks/baboondiversity/data/PG_panu3_zarr_12_03_2021/callset.zarr/chrX"
#Opening the zarr data
callset_f = zarr.open_group(zarr_chrX_dir, mode="r")
gt_f = allel.GenotypeArray(callset_f["calldata/GT"])
pos_f = callset_f["variants/POS"][:]

zarr_all_chrX_dir = "/home/eriks/baboondiversity/data/PG_panu3_zarr_12_03_2021/callset.zarr/all_chrX"
#Opening the zarr data
callset_all_chrX = zarr.open_group(zarr_all_chrX_dir, mode="r")
gt_all_chrX = allel.GenotypeArray(callset_all_chrX["calldata/GT"])
pos_all_chrX = callset_all_chrX["variants/POS"][:]

zarr_dipmale_chrX = "/home/eriks/baboondiversity/data/PG_panu3_zarr_12_03_2021/callset.zarr/dipmale_chrX"
#Opening the zarr data
callset_dipmale_chrX = zarr.open_group(zarr_dipmale_chrX, mode="r")

meta_data_samples_sci = pd.read_csv("../data/Papio_metadata_with_clustering_sci.txt", sep =" ")
meta_data_samples = pd.read_csv("../data/Papio_metadata_with_clustering.txt", sep =" ")

#Generate a mapping between metadata and callset - repeat for chrX females to be sure of no errors.
ID_to_callset_f = dict(zip(callset_f["samples"][:], range(len(callset_f["samples"][:]))))
meta_data_samples_f = meta_data_samples_sci.loc[meta_data_samples_sci.PGDP_ID.isin(callset_f["samples"][:])].copy()
meta_data_samples_f["callset_index"] = meta_data_samples_f.PGDP_ID.map(ID_to_callset_f)

ID_to_callset_dip = dict(zip(callset_dipmale_chrX["samples"][:], range(len(callset_dipmale_chrX["samples"][:]))))
meta_data_samples_dip = meta_data_samples_sci.loc[meta_data_samples_sci.PGDP_ID.isin(callset_dipmale_chrX["samples"][:])].copy()
meta_data_samples_dip["callset_index"] = meta_data_samples_dip.PGDP_ID.map(ID_to_callset_dip)

In [4]:
rfmix_path = "../steps/rfmix_gen100/eth_olive_focus/"
df_l = []
chroms = ["chr{}".format(x) for x in (range(1, 21))]+["all_chrX"]
for c in chroms:
    read_file = rfmix_path + "{}.windows.txt".format(c)
    df = pd.read_csv(read_file, sep="\t")
    df_l.append(df)
window_df_eth = pd.concat(df_l)
mean_window_df_eth = window_df_eth.groupby(["chrom", "individual", "start", "end"]).mean().reset_index()

In [6]:
rfmix_path = "../steps/rfmix_gen100/tanzania_gm_focus/"

df_l = []
chroms = ["chr{}".format(x) for x in (range(1, 21))]+["all_chrX"]
for c in chroms:
    read_file = rfmix_path + "{}.windows.txt".format(c)
    df = pd.read_csv(read_file, sep="\t")
    df_l.append(df)
window_df_tanz = pd.concat(df_l)
mean_window_df_tanz = window_df_tanz.groupby(["chrom", "individual", "start", "end"]).mean().reset_index()
tanz_olives = meta_data_samples_sci.loc[meta_data_samples_sci.C_origin == "Anubis, Tanzania"].PGDP_ID
mean_window_df_tanz_olive = mean_window_df_tanz.loc[mean_window_df_tanz.individual.isin(tanz_olives)]

In [ ]:
meta_data_samples.Origin.unique()

In [ ]:
focus_pop_tags = ["Chunga, Zambia", "Dendro Park, Zambia", "Niokolo-Koba, Senegal", "Filoha, Ethiopia"]
window_size = 100000
all_chroms = ["chr{}".format(x) for x in (range(1, 21))]+["chrX"]

c_df_l = []
for c in all_chroms:
    print(c)
    zarr_dir = "/home/eriks/baboondiversity/data/PG_panu3_zarr_12_03_2021/callset.zarr/" + c
    #Opening the zarr data
    callset = zarr.open_group(zarr_dir, mode="r")
    gt = allel.GenotypeArray(callset["calldata/GT"])
    pos = callset["variants/POS"][:]
    # Loading in the IDs and gt
    df_l = []
    for p in focus_pop_tags:
        if c == "chrX":
            metadata = meta_data_samples_f
            focus_pop_gt = gt.take(metadata.loc[(metadata.Origin == p) &
                                            (metadata.Sex == "F")].callset_index,
                       axis=1)
        else:
            metadata = meta_data_samples
            focus_pop_gt = gt.take(metadata.loc[metadata.Origin == p].callset_index,
                       axis=1)
        pi, windows, n_bases, counts = allel.windowed_diversity(pos, focus_pop_gt.count_alleles(),
                          size=window_size, start=0)
        df_l.append(pd.DataFrame({"chrom": c, "population": p, "start": windows[:,0], "diversity": pi}))
    focus_df = pd.concat(df_l)
    if c == "chrX":
        rf_c = "female_chrX"
    elif c == "dipmale_chrX":
        rf_c = "all_chrX"
    else:
        rf_c = c
    print(c, rf_c)
    c_df_l.append(focus_df)

In [ ]:
diversity_df = pd.concat(c_df_l)
mean_diversity = diversity_df.groupby(["chrom", "start"])[["diversity"]].mean().reset_index()
mean_diversity

I split based on origin.

In [ ]:
window_df_tanz["Origin"] = window_df_tanz.individual.map(dict(zip(meta_data_samples_sci.PGDP_ID, meta_data_samples_sci.Origin)))

In [ ]:
window_df_eth["Origin"] = window_df_eth.individual.map(dict(zip(meta_data_samples_sci.PGDP_ID, meta_data_samples_sci.Origin+"_eth_case")))

In [ ]:
window_df_tanz_eth = pd.concat([window_df_tanz, window_df_eth])

In [ ]:
mean_window_df_tanz_eth = window_df_tanz_eth.groupby(["chrom", "Origin", "start", "end"])[["north_sum"]].mean().reset_index()
mean_window_df_tanz = window_df_tanz.groupby(["chrom", "Origin", "start", "end"])[["north_sum"]].mean().reset_index()
admix_div_mean = mean_diversity.merge(mean_window_df_tanz_eth, on=["chrom", "start"])
admix_div_mean["Species"] = admix_div_mean.Origin.map(dict(zip(meta_data_samples_sci.Origin, meta_data_samples_sci.Species)))
admix_div_mean = admix_div_mean.merge(c_r_g_df, on=["chrom", "start"])
admix_div_mean = admix_div_mean.loc[admix_div_mean.chrom != "all_chrX"]
admix_div_mean["North Percentage"] = admix_div_mean.north_sum/100000
admix_div_mean["logdiv"] = np.log10(admix_div_mean.diversity)
admix_div_mean["logrecomb"] = np.log10(admix_div_mean.average_cM_window)
admix_div_mean["norm_diversity"] = (admix_div_mean.diversity-admix_div_mean.diversity.mean())/admix_div_mean.diversity.std()
admix_div_mean["norm_recomb"] = (admix_div_mean.average_cM_window-admix_div_mean.average_cM_window.mean())/admix_div_mean.average_cM_window.std()

In [ ]:
#Selecting the cases of interest and setting Minor Parent Ancestry
origins_interest = ['Arusha, Tanzania', 'Mahale, Tanzania',
       'Gog Woreda, Gambella region, Ethiopia_eth_case',
       'Gombe, Tanzania', 'Issa Valley, Tanzania',
       'Katavi, Tanzania','Lake Manyara, Tanzania',
       'Ngorongoro, Tanzania', 'Ruaha, Tanzania',
       'Selous, Tanzania', 'Serengeti, Tanzania', 'Tarangire, Tanzania',
       'Udzungwa, Tanzania']
admix_div_mean = admix_div_mean.loc[admix_div_mean.Origin.isin(origins_interest)]
admix_div_mean["minor_parent_percentage"] = [x if y == ("cynocephalus") or z == "Gog Woreda, Gambella region, Ethiopia_eth_case"
                                             else 1-x for x, y, z in zip(admix_div_mean["North Percentage"],
                                                                         admix_div_mean["Species"],
                                                                        admix_div_mean["Origin"])]
admix_div_mean["local_minor_ancestry"] = [min(x, 1-x) for x in admix_div_mean["North Percentage"]]

In [ ]:
yellows = ['Mahale, Tanzania', 'Katavi, Tanzania', 'Ruaha, Tanzania']
olives = ['Serengeti, Tanzania', 'Ngorongoro, Tanzania', 'Gombe, Tanzania',
          'Lake Manyara, Tanzania','Arusha, Tanzania']
olives_with_tarangire = ['Serengeti, Tanzania', 'Tarangire, Tanzania', 'Ngorongoro, Tanzania', 'Gombe, Tanzania',
          'Lake Manyara, Tanzania','Arusha, Tanzania']
gog = ['Gog Woreda, Ethiopia']
result_order = ['Mahale, Tanzania', 'Katavi, Tanzania', 'Ruaha, Tanzania',
       'Tarangire, Tanzania', 'Arusha, Tanzania', 'Ngorongoro, Tanzania', 'Gombe, Tanzania', 
             'Lake Manyara, Tanzania', 'Serengeti, Tanzania']
result_order_no_gog = ['Mahale, Tanzania', 'Katavi, Tanzania', 'Ruaha, Tanzania',
       'Tarangire, Tanzania', 'Arusha, Tanzania', 'Ngorongoro, Tanzania', 'Gombe, Tanzania', 
             'Lake Manyara, Tanzania', 'Serengeti, Tanzania']

In [ ]:
import copy
from textwrap import dedent
import warnings
import numpy as np
import pandas as pd
import matplotlib as mpl
import matplotlib.pyplot as plt

try:
    import statsmodels
    assert statsmodels
    _has_statsmodels = True
except ImportError:
    _has_statsmodels = False

from seaborn import utils
from seaborn import algorithms as algo
from seaborn.axisgrid import FacetGrid, _facet_docs


__all__ = ["lmplot", "regplot", "residplot"]


class _LinearPlotter:
    """Base class for plotting relational data in tidy format.

    To get anything useful done you'll have to inherit from this, but setup
    code that can be abstracted out should be put here.

    """
    def establish_variables(self, data, **kws):
        """Extract variables from data or use directly."""
        self.data = data

        # Validate the inputs
        any_strings = any([isinstance(v, str) for v in kws.values()])
        if any_strings and data is None:
            raise ValueError("Must pass `data` if using named variables.")

        # Set the variables
        for var, val in kws.items():
            if isinstance(val, str):
                vector = data[val]
            elif isinstance(val, list):
                vector = np.asarray(val)
            else:
                vector = val
            if vector is not None and vector.shape != (1,):
                vector = np.squeeze(vector)
            if np.ndim(vector) > 1:
                err = "regplot inputs must be 1d"
                raise ValueError(err)
            setattr(self, var, vector)

    def dropna(self, *vars):
        """Remove observations with missing data."""
        vals = [getattr(self, var) for var in vars]
        vals = [v for v in vals if v is not None]
        not_na = np.all(np.column_stack([pd.notnull(v) for v in vals]), axis=1)
        for var in vars:
            val = getattr(self, var)
            if val is not None:
                setattr(self, var, val[not_na])

    def plot(self, ax):
        raise NotImplementedError


class _RegressionPlotter(_LinearPlotter):
    """Plotter for numeric independent variables with regression model.

    This does the computations and drawing for the `regplot` function, and
    is thus also used indirectly by `lmplot`.
    """
    def __init__(self, x, y, data=None, x_estimator=None, x_bins=None,
                 x_ci="ci", scatter=True, fit_reg=True, ci=95, n_boot=1000,
                 units=None, seed=None, order=1, logistic=False, lowess=False,
                 robust=False, weighted=False, logx=False, x_partial=None, y_partial=None,
                 truncate=False, dropna=True, x_jitter=None, y_jitter=None,
                 color=None, label=None):

        # Set member attributes
        self.x_estimator = x_estimator
        self.ci = ci
        self.x_ci = ci if x_ci == "ci" else x_ci
        self.n_boot = n_boot
        self.seed = seed
        self.scatter = scatter
        self.fit_reg = fit_reg
        self.order = order
        self.logistic = logistic
        self.weighted = weighted
        self.lowess = lowess
        self.robust = robust
        self.logx = logx
        self.truncate = truncate
        self.x_jitter = x_jitter
        self.y_jitter = y_jitter
        self.color = color
        self.label = label

        # Validate the regression options:
        #if sum((order > 1, logistic, robust, weighted, lowess, logx)) > 1:
        #    raise ValueError("Mutually exclusive regression options.")

        # Extract the data vals from the arguments or passed dataframe
        self.establish_variables(data, x=x, y=y, units=units,
                                 x_partial=x_partial, y_partial=y_partial)

        # Drop null observations
        if dropna:
            self.dropna("x", "y", "units", "x_partial", "y_partial")

        # Regress nuisance variables out of the data
        if self.x_partial is not None:
            self.x = self.regress_out(self.x, self.x_partial)
        if self.y_partial is not None:
            self.y = self.regress_out(self.y, self.y_partial)

        # Possibly bin the predictor variable, which implies a point estimate
        if x_bins is not None:
            self.x_estimator = np.mean if x_estimator is None else x_estimator
            x_discrete, x_bins = self.bin_predictor(x_bins)
            self.x_discrete = x_discrete
        else:
            self.x_discrete = self.x

        # Disable regression in case of singleton inputs
        if len(self.x) <= 1:
            self.fit_reg = False

        # Save the range of the x variable for the grid later
        if self.fit_reg:
            self.x_range = self.x.min(), self.x.max()

    @property
    def scatter_data(self):
        """Data where each observation is a point."""
        x_j = self.x_jitter
        if x_j is None:
            x = self.x
        else:
            x = self.x + np.random.uniform(-x_j, x_j, len(self.x))

        y_j = self.y_jitter
        if y_j is None:
            y = self.y
        else:
            y = self.y + np.random.uniform(-y_j, y_j, len(self.y))

        return x, y

    @property
    def estimate_data(self):
        """Data with a point estimate and CI for each discrete x value."""
        x, y = self.x_discrete, self.y
        vals = sorted(np.unique(x))
        points, cis = [], []

        for val in vals:

            # Get the point estimate of the y variable
            _y = y[x == val]
            est = self.x_estimator(_y)
            points.append(est)

            # Compute the confidence interval for this estimate
            if self.x_ci is None:
                cis.append(None)
            else:
                units = None
                if self.x_ci == "sd":
                    sd = np.std(_y)
                    _ci = est - sd, est + sd
                else:
                    if self.units is not None:
                        units = self.units[x == val]
                    boots = algo.bootstrap(_y,
                                           func=self.x_estimator,
                                           n_boot=self.n_boot,
                                           units=units,
                                           seed=self.seed)
                    _ci = utils.ci(boots, self.x_ci)
                cis.append(_ci)

        return vals, points, cis

    def _check_statsmodels(self):
        """Check whether statsmodels is installed if any boolean options require it."""
        options = "logistic", "robust", "lowess"
        err = "`{}=True` requires statsmodels, an optional dependency, to be installed."
        for option in options:
            if getattr(self, option) and not _has_statsmodels:
                raise RuntimeError(err.format(option))

    def fit_regression(self, ax=None, x_range=None, grid=None):
        """Fit the regression model."""
        self._check_statsmodels()

        # Create the grid for the regression
        if grid is None:
            if self.truncate:
                x_min, x_max = self.x_range
            else:
                if ax is None:
                    x_min, x_max = x_range
                else:
                    x_min, x_max = ax.get_xlim()
            grid = np.linspace(x_min, x_max, 100)
        ci = self.ci

        # Fit the regression
        if self.order > 1:
            yhat, yhat_boots = self.fit_poly(grid, self.order)
        elif self.logistic: 
            from statsmodels.genmod.generalized_linear_model import GLM
            from statsmodels.genmod.families import Binomial
            yhat, yhat_boots = self.fit_statsmodels(grid, GLM,
                                                    family=Binomial())
        elif self.weighted: 
            from statsmodels.genmod.generalized_linear_model import GLM
            from statsmodels.genmod.families import Binomial
            yhat, yhat_boots = self.fit_statsmodels(grid, GLM,
                                                   var_weights=np.asarray(self.x))
        elif self.lowess:
            ci = None
            grid, yhat = self.fit_lowess()
        elif self.robust:
            from statsmodels.robust.robust_linear_model import RLM
            yhat, yhat_boots = self.fit_statsmodels(grid, RLM)
        elif self.logx:
            yhat, yhat_boots = self.fit_logx(grid)
        else:
            yhat, yhat_boots = self.fit_fast(grid)

        # Compute the confidence interval at each grid point
        if ci is None:
            err_bands = None
        else:
            err_bands = utils.ci(yhat_boots, ci, axis=0)

        return grid, yhat, err_bands

    def fit_fast(self, grid):
        """Low-level regression and prediction using linear algebra."""
        def reg_func(_x, _y):
            return np.linalg.pinv(_x).dot(_y)

        X, y = np.c_[np.ones(len(self.x)), self.x], self.y
        grid = np.c_[np.ones(len(grid)), grid]
        yhat = grid.dot(reg_func(X, y))
        if self.ci is None:
            return yhat, None

        beta_boots = algo.bootstrap(X, y,
                                    func=reg_func,
                                    n_boot=self.n_boot,
                                    units=self.units,
                                    seed=self.seed).T
        yhat_boots = grid.dot(beta_boots).T
        return yhat, yhat_boots

    def fit_poly(self, grid, order):
        """Regression using numpy polyfit for higher-order trends."""
        def reg_func(_x, _y):
            return np.polyval(np.polyfit(_x, _y, order), grid)

        x, y = self.x, self.y
        yhat = reg_func(x, y)
        if self.ci is None:
            return yhat, None

        yhat_boots = algo.bootstrap(x, y,
                                    func=reg_func,
                                    n_boot=self.n_boot,
                                    units=self.units,
                                    seed=self.seed)
        return yhat, yhat_boots

    def fit_statsmodels(self, grid, model, **kwargs):
        """More general regression function using statsmodels objects."""
        import statsmodels.tools.sm_exceptions as sme
        X, y = np.c_[np.ones(len(self.x)), self.x], self.y
        grid = np.c_[np.ones(len(grid)), grid]
        def reg_func(_x, _y, **kwargs):
            err_classes = (sme.PerfectSeparationError,)
            try:
                with warnings.catch_warnings():
                    if hasattr(sme, "PerfectSeparationWarning"):
                        # statsmodels>=0.14.0
                        warnings.simplefilter("error", sme.PerfectSeparationWarning)
                        err_classes = (*err_classes, sme.PerfectSeparationWarning)
                    m = model(_y, _x, var_weights=np.asarray(_x[:, 1])).fit()
                    yhat = m.predict(grid)
                    #print(m.summary(), self.x, len(_x), _x[:, 1])
            except err_classes:
                yhat = np.empty(len(grid))
                yhat.fill(np.nan)
            return yhat

        yhat = reg_func(X, y, **kwargs)
        if self.ci is None:
            return yhat, None

        yhat_boots = bootstrap(X, y,
                                    func=reg_func,
                                    n_boot=self.n_boot,
                                    units=self.units,
                                    seed=self.seed,
                                   var_weights=np.asarray(self.x))
        return yhat, yhat_boots

    def fit_lowess(self):
        """Fit a locally-weighted regression, which returns its own grid."""
        from statsmodels.nonparametric.smoothers_lowess import lowess
        grid, yhat = lowess(self.y, self.x).T
        return grid, yhat

    def fit_logx(self, grid):
        """Fit the model in log-space."""
        X, y = np.c_[np.ones(len(self.x)), self.x], self.y
        grid = np.c_[np.ones(len(grid)), np.log(grid)]

        def reg_func(_x, _y):
            _x = np.c_[_x[:, 0], np.log(_x[:, 1])]
            return np.linalg.pinv(_x).dot(_y)

        yhat = grid.dot(reg_func(X, y))
        if self.ci is None:
            return yhat, None

        beta_boots = algo.bootstrap(X, y,
                                    func=reg_func,
                                    n_boot=self.n_boot,
                                    units=self.units,
                                    seed=self.seed).T
        yhat_boots = grid.dot(beta_boots).T
        return yhat, yhat_boots

    def bin_predictor(self, bins):
        """Discretize a predictor by assigning value to closest bin."""
        x = np.asarray(self.x)
        if np.isscalar(bins):
            percentiles = np.linspace(0, 100, bins + 2)[1:-1]
            bins = np.percentile(x, percentiles)
        else:
            bins = np.ravel(bins)

        dist = np.abs(np.subtract.outer(x, bins))
        x_binned = bins[np.argmin(dist, axis=1)].ravel()

        return x_binned, bins

    def regress_out(self, a, b):
        """Regress b from a keeping a's original mean."""
        a_mean = a.mean()
        a = a - a_mean
        b = b - b.mean()
        b = np.c_[b]
        a_prime = a - b.dot(np.linalg.pinv(b).dot(a))
        return np.asarray(a_prime + a_mean).reshape(a.shape)

    def plot(self, ax, scatter_kws, line_kws):
        """Draw the full plot."""
        # Insert the plot label into the correct set of keyword arguments
        if self.scatter:
            scatter_kws["label"] = self.label
        else:
            line_kws["label"] = self.label

        # Use the current color cycle state as a default
        if self.color is None:
            lines, = ax.plot([], [])
            color = lines.get_color()
            lines.remove()
        else:
            color = self.color

        # Ensure that color is hex to avoid matplotlib weirdness
        color = mpl.colors.rgb2hex(mpl.colors.colorConverter.to_rgb(color))

        # Let color in keyword arguments override overall plot color
        scatter_kws.setdefault("color", color)
        line_kws.setdefault("color", color)

        # Draw the constituent plots
        if self.scatter:
            self.scatterplot(ax, scatter_kws)

        if self.fit_reg:
            self.lineplot(ax, line_kws)

        # Label the axes
        if hasattr(self.x, "name"):
            ax.set_xlabel(self.x.name)
        if hasattr(self.y, "name"):
            ax.set_ylabel(self.y.name)

    def scatterplot(self, ax, kws):
        """Draw the data."""
        # Treat the line-based markers specially, explicitly setting larger
        # linewidth than is provided by the seaborn style defaults.
        # This would ideally be handled better in matplotlib (i.e., distinguish
        # between edgewidth for solid glyphs and linewidth for line glyphs
        # but this should do for now.
        line_markers = ["1", "2", "3", "4", "+", "x", "|", "_"]
        if self.x_estimator is None:
            if "marker" in kws and kws["marker"] in line_markers:
                lw = mpl.rcParams["lines.linewidth"]
            else:
                lw = mpl.rcParams["lines.markeredgewidth"]
            kws.setdefault("linewidths", lw)

            if not hasattr(kws['color'], 'shape') or kws['color'].shape[1] < 4:
                kws.setdefault("alpha", .8)

            x, y = self.scatter_data
            ax.scatter(x, y, **kws)
        else:
            # TODO abstraction
            ci_kws = {"color": kws["color"]}
            if "alpha" in kws:
                ci_kws["alpha"] = kws["alpha"]
            ci_kws["linewidth"] = mpl.rcParams["lines.linewidth"] * 1.75
            kws.setdefault("s", 50)

            xs, ys, cis = self.estimate_data
            if [ci for ci in cis if ci is not None]:
                for x, ci in zip(xs, cis):
                    ax.plot([x, x], ci, **ci_kws)
            ax.scatter(xs, ys, **kws)

    def lineplot(self, ax, kws):
        """Draw the model."""
        # Fit the regression model
        grid, yhat, err_bands = self.fit_regression(ax)
        edges = grid[0], grid[-1]

        # Get set default aesthetics
        fill_color = kws["color"]
        lw = kws.pop("lw", mpl.rcParams["lines.linewidth"] * 1.5)
        kws.setdefault("linewidth", lw)

        # Draw the regression line and confidence interval
        line, = ax.plot(grid, yhat, **kws)
        if not self.truncate:
            line.sticky_edges.x[:] = edges  # Prevent mpl from adding margin
        if err_bands is not None:
            ax.fill_between(grid, *err_bands, facecolor=fill_color, alpha=.15)


_regression_docs = dict(

    model_api=dedent("""\
    There are a number of mutually exclusive options for estimating the
    regression model. See the :ref:`tutorial <regression_tutorial>` for more
    information.\
    """),
    regplot_vs_lmplot=dedent("""\
    The :func:`regplot` and :func:`lmplot` functions are closely related, but
    the former is an axes-level function while the latter is a figure-level
    function that combines :func:`regplot` and :class:`FacetGrid`.\
    """),
    x_estimator=dedent("""\
    x_estimator : callable that maps vector -> scalar, optional
        Apply this function to each unique value of ``x`` and plot the
        resulting estimate. This is useful when ``x`` is a discrete variable.
        If ``x_ci`` is given, this estimate will be bootstrapped and a
        confidence interval will be drawn.\
    """),
    x_bins=dedent("""\
    x_bins : int or vector, optional
        Bin the ``x`` variable into discrete bins and then estimate the central
        tendency and a confidence interval. This binning only influences how
        the scatterplot is drawn; the regression is still fit to the original
        data.  This parameter is interpreted either as the number of
        evenly-sized (not necessary spaced) bins or the positions of the bin
        centers. When this parameter is used, it implies that the default of
        ``x_estimator`` is ``numpy.mean``.\
    """),
    x_ci=dedent("""\
    x_ci : "ci", "sd", int in [0, 100] or None, optional
        Size of the confidence interval used when plotting a central tendency
        for discrete values of ``x``. If ``"ci"``, defer to the value of the
        ``ci`` parameter. If ``"sd"``, skip bootstrapping and show the
        standard deviation of the observations in each bin.\
    """),
    scatter=dedent("""\
    scatter : bool, optional
        If ``True``, draw a scatterplot with the underlying observations (or
        the ``x_estimator`` values).\
    """),
    fit_reg=dedent("""\
    fit_reg : bool, optional
        If ``True``, estimate and plot a regression model relating the ``x``
        and ``y`` variables.\
    """),
    ci=dedent("""\
    ci : int in [0, 100] or None, optional
        Size of the confidence interval for the regression estimate. This will
        be drawn using translucent bands around the regression line. The
        confidence interval is estimated using a bootstrap; for large
        datasets, it may be advisable to avoid that computation by setting
        this parameter to None.\
    """),
    n_boot=dedent("""\
    n_boot : int, optional
        Number of bootstrap resamples used to estimate the ``ci``. The default
        value attempts to balance time and stability; you may want to increase
        this value for "final" versions of plots.\
    """),
    units=dedent("""\
    units : variable name in ``data``, optional
        If the ``x`` and ``y`` observations are nested within sampling units,
        those can be specified here. This will be taken into account when
        computing the confidence intervals by performing a multilevel bootstrap
        that resamples both units and observations (within unit). This does not
        otherwise influence how the regression is estimated or drawn.\
    """),
    seed=dedent("""\
    seed : int, numpy.random.Generator, or numpy.random.RandomState, optional
        Seed or random number generator for reproducible bootstrapping.\
    """),
    order=dedent("""\
    order : int, optional
        If ``order`` is greater than 1, use ``numpy.polyfit`` to estimate a
        polynomial regression.\
    """),
    logistic=dedent("""\
    logistic : bool, optional
        If ``True``, assume that ``y`` is a binary variable and use
        ``statsmodels`` to estimate a logistic regression model. Note that this
        is substantially more computationally intensive than linear regression,
        so you may wish to decrease the number of bootstrap resamples
        (``n_boot``) or set ``ci`` to None.\
    """),
    lowess=dedent("""\
    lowess : bool, optional
        If ``True``, use ``statsmodels`` to estimate a nonparametric lowess
        model (locally weighted linear regression). Note that confidence
        intervals cannot currently be drawn for this kind of model.\
    """),
    robust=dedent("""\
    robust : bool, optional
        If ``True``, use ``statsmodels`` to estimate a robust regression. This
        will de-weight outliers. Note that this is substantially more
        computationally intensive than standard linear regression, so you may
        wish to decrease the number of bootstrap resamples (``n_boot``) or set
        ``ci`` to None.\
    """),
    logx=dedent("""\
    logx : bool, optional
        If ``True``, estimate a linear regression of the form y ~ log(x), but
        plot the scatterplot and regression model in the input space. Note that
        ``x`` must be positive for this to work.\
    """),
    xy_partial=dedent("""\
    {x,y}_partial : strings in ``data`` or matrices
        Confounding variables to regress out of the ``x`` or ``y`` variables
        before plotting.\
    """),
    truncate=dedent("""\
    truncate : bool, optional
        If ``True``, the regression line is bounded by the data limits. If
        ``False``, it extends to the ``x`` axis limits.
    """),
    dropna=dedent("""\
    dropna : bool, optional
        If ``True``, remove observations with missing data from the plot.
    """),
    xy_jitter=dedent("""\
    {x,y}_jitter : floats, optional
        Add uniform random noise of this size to either the ``x`` or ``y``
        variables. The noise is added to a copy of the data after fitting the
        regression, and only influences the look of the scatterplot. This can
        be helpful when plotting variables that take discrete values.\
    """),
    scatter_line_kws=dedent("""\
    {scatter,line}_kws : dictionaries
        Additional keyword arguments to pass to ``plt.scatter`` and
        ``plt.plot``.\
    """),
)
_regression_docs.update(_facet_docs)


def lmplot(
    data, *,
    x=None, y=None, hue=None, col=None, row=None,
    palette=None, col_wrap=None, height=5, aspect=1, markers="o",
    sharex=None, sharey=None, hue_order=None, col_order=None, row_order=None,
    legend=True, legend_out=None, x_estimator=None, x_bins=None,
    x_ci="ci", scatter=True, fit_reg=True, ci=95, n_boot=1000,
    units=None, seed=None, order=1, logistic=False, lowess=False,
    robust=False, weighted=False, logx=False, x_partial=None, y_partial=None,
    truncate=True, x_jitter=None, y_jitter=None, scatter_kws=None,
    line_kws=None, facet_kws=None,
):

    if facet_kws is None:
        facet_kws = {}

    def facet_kw_deprecation(key, val):
        msg = (
            f"{key} is deprecated from the `lmplot` function signature. "
            "Please update your code to pass it using `facet_kws`."
        )
        if val is not None:
            warnings.warn(msg, UserWarning)
            facet_kws[key] = val

    facet_kw_deprecation("sharex", sharex)
    facet_kw_deprecation("sharey", sharey)
    facet_kw_deprecation("legend_out", legend_out)

    if data is None:
        raise TypeError("Missing required keyword argument `data`.")

    # Reduce the dataframe to only needed columns
    need_cols = [x, y, hue, col, row, units, x_partial, y_partial]
    cols = np.unique([a for a in need_cols if a is not None]).tolist()
    data = data[cols]

    # Initialize the grid
    facets = FacetGrid(
        data, row=row, col=col, hue=hue,
        palette=palette,
        row_order=row_order, col_order=col_order, hue_order=hue_order,
        height=height, aspect=aspect, col_wrap=col_wrap,
        **facet_kws,
    )

    # Add the markers here as FacetGrid has figured out how many levels of the
    # hue variable are needed and we don't want to duplicate that process
    if facets.hue_names is None:
        n_markers = 1
    else:
        n_markers = len(facets.hue_names)
    if not isinstance(markers, list):
        markers = [markers] * n_markers
    if len(markers) != n_markers:
        raise ValueError("markers must be a singleton or a list of markers "
                         "for each level of the hue variable")
    facets.hue_kws = {"marker": markers}

    #def update_datalim(data, x, y, ax, **kws):
    #    xys = data[[x, y]].to_numpy().astype(float)
    #    ax.update_datalim(xys, updatey=False)
    #    ax.autoscale_view(scaley=False)

    #facets.map_dataframe(update_datalim, x=x, y=y)

    # Draw the regression plot on each facet
    regplot_kws = dict(
        x_estimator=x_estimator, x_bins=x_bins, x_ci=x_ci,
        scatter=scatter, fit_reg=fit_reg, ci=ci, n_boot=n_boot, units=units,
        seed=seed, order=order, logistic=logistic, lowess=lowess,
        robust=robust, weighted=weighted, logx=logx, x_partial=x_partial, y_partial=y_partial,
        truncate=truncate, x_jitter=x_jitter, y_jitter=y_jitter,
        scatter_kws=scatter_kws, line_kws=line_kws,
    )
    facets.map_dataframe(regplot, x=x, y=y, **regplot_kws)
    facets.set_axis_labels(x, y)

    # Add a legend
    if legend and (hue is not None) and (hue not in [col, row]):
        facets.add_legend()
    return facets


lmplot.__doc__ = dedent("""\
    Plot data and regression model fits across a FacetGrid.

    This function combines :func:`regplot` and :class:`FacetGrid`. It is
    intended as a convenient interface to fit regression models across
    conditional subsets of a dataset.

    When thinking about how to assign variables to different facets, a general
    rule is that it makes sense to use ``hue`` for the most important
    comparison, followed by ``col`` and ``row``. However, always think about
    your particular dataset and the goals of the visualization you are
    creating.

    {model_api}

    The parameters to this function span most of the options in
    :class:`FacetGrid`, although there may be occasional cases where you will
    want to use that class and :func:`regplot` directly.

    Parameters
    ----------
    {data}
    x, y : strings, optional
        Input variables; these should be column names in ``data``.
    hue, col, row : strings
        Variables that define subsets of the data, which will be drawn on
        separate facets in the grid. See the ``*_order`` parameters to control
        the order of levels of this variable.
    {palette}
    {col_wrap}
    {height}
    {aspect}
    markers : matplotlib marker code or list of marker codes, optional
        Markers for the scatterplot. If a list, each marker in the list will be
        used for each level of the ``hue`` variable.
    {share_xy}

        .. deprecated:: 0.12.0
            Pass using the `facet_kws` dictionary.

    {{hue,col,row}}_order : lists, optional
        Order for the levels of the faceting variables. By default, this will
        be the order that the levels appear in ``data`` or, if the variables
        are pandas categoricals, the category order.
    legend : bool, optional
        If ``True`` and there is a ``hue`` variable, add a legend.
    {legend_out}

        .. deprecated:: 0.12.0
            Pass using the `facet_kws` dictionary.

    {x_estimator}
    {x_bins}
    {x_ci}
    {scatter}
    {fit_reg}
    {ci}
    {n_boot}
    {units}
    {seed}
    {order}
    {logistic}
    {lowess}
    {robust}
    {logx}
    {xy_partial}
    {truncate}
    {xy_jitter}
    {scatter_line_kws}
    facet_kws : dict
        Dictionary of keyword arguments for :class:`FacetGrid`.

    Returns
    -------
    :class:`FacetGrid`
        The :class:`FacetGrid` object with the plot on it for further tweaking.

    See Also
    --------
    regplot : Plot data and a conditional model fit.
    FacetGrid : Subplot grid for plotting conditional relationships.
    pairplot : Combine :func:`regplot` and :class:`PairGrid` (when used with
               ``kind="reg"``).

    Notes
    -----

    {regplot_vs_lmplot}

    Examples
    --------

    .. include:: ../docstrings/lmplot.rst

    """).format(**_regression_docs)


def regplot(
    data=None, *, x=None, y=None,
    x_estimator=None, x_bins=None, x_ci="ci",
    scatter=True, fit_reg=True, ci=95, n_boot=1000, units=None,
    seed=None, order=1, logistic=False, lowess=False, robust=False, weighted=False,
    logx=False, x_partial=None, y_partial=None,
    truncate=True, dropna=True, x_jitter=None, y_jitter=None,
    label=None, color=None, marker="o",
    scatter_kws=None, line_kws=None, ax=None
):

    plotter = _RegressionPlotter(x, y, data, x_estimator, x_bins, x_ci,
                                 scatter, fit_reg, ci, n_boot, units, seed,
                                 order, logistic, lowess, robust, weighted, logx,
                                 x_partial, y_partial, truncate, dropna,
                                 x_jitter, y_jitter, color, label)

    if ax is None:
        ax = plt.gca()

    scatter_kws = {} if scatter_kws is None else copy.copy(scatter_kws)
    scatter_kws["marker"] = marker
    line_kws = {} if line_kws is None else copy.copy(line_kws)
    plotter.plot(ax, scatter_kws, line_kws)
    return ax


regplot.__doc__ = dedent("""\
    Plot data and a linear regression model fit.

    {model_api}

    Parameters
    ----------
    x, y : string, series, or vector array
        Input variables. If strings, these should correspond with column names
        in ``data``. When pandas objects are used, axes will be labeled with
        the series name.
    {data}
    {x_estimator}
    {x_bins}
    {x_ci}
    {scatter}
    {fit_reg}
    {ci}
    {n_boot}
    {units}
    {seed}
    {order}
    {logistic}
    {lowess}
    {robust}
    {logx}
    {xy_partial}
    {truncate}
    {dropna}
    {xy_jitter}
    label : string
        Label to apply to either the scatterplot or regression line (if
        ``scatter`` is ``False``) for use in a legend.
    color : matplotlib color
        Color to apply to all plot elements; will be superseded by colors
        passed in ``scatter_kws`` or ``line_kws``.
    marker : matplotlib marker code
        Marker to use for the scatterplot glyphs.
    {scatter_line_kws}
    ax : matplotlib Axes, optional
        Axes object to draw the plot onto, otherwise uses the current Axes.

    Returns
    -------
    ax : matplotlib Axes
        The Axes object containing the plot.

    See Also
    --------
    lmplot : Combine :func:`regplot` and :class:`FacetGrid` to plot multiple
             linear relationships in a dataset.
    jointplot : Combine :func:`regplot` and :class:`JointGrid` (when used with
                ``kind="reg"``).
    pairplot : Combine :func:`regplot` and :class:`PairGrid` (when used with
               ``kind="reg"``).
    residplot : Plot the residuals of a linear regression model.

    Notes
    -----

    {regplot_vs_lmplot}


    It's also easy to combine :func:`regplot` and :class:`JointGrid` or
    :class:`PairGrid` through the :func:`jointplot` and :func:`pairplot`
    functions, although these do not directly accept all of :func:`regplot`'s
    parameters.

    Examples
    --------

    .. include:: ../docstrings/regplot.rst

    """).format(**_regression_docs)


def bootstrap(*args, **kwargs):
    """Resample one or more arrays with replacement and store aggregate values.

    Positional arguments are a sequence of arrays to bootstrap along the first
    axis and pass to a summary function.

    Keyword arguments:
        n_boot : int, default=10000
            Number of iterations
        axis : int, default=None
            Will pass axis to ``func`` as a keyword argument.
        units : array, default=None
            Array of sampling unit IDs. When used the bootstrap resamples units
            and then observations within units instead of individual
            datapoints.
        func : string or callable, default="mean"
            Function to call on the args that are passed in. If string, uses as
            name of function in the numpy namespace. If nans are present in the
            data, will try to use nan-aware version of named function.
        seed : Generator | SeedSequence | RandomState | int | None
            Seed for the random number generator; useful if you want
            reproducible resamples.

    Returns
    -------
    boot_dist: array
        array of bootstrapped statistic values

    """
    # Ensure list of arrays are same length
    if len(np.unique(list(map(len, args)))) > 1:
        raise ValueError("All input arrays must have the same length")
    n = len(args[0])

    # Default keyword arguments
    n_boot = kwargs.get("n_boot", 10000)
    func = kwargs.get("func", "mean")
    axis = kwargs.get("axis", None)
    units = kwargs.get("units", None)
    var_weights = kwargs.get("var_weights", None)
    random_seed = kwargs.get("random_seed", None)
    if random_seed is not None:
        msg = "`random_seed` has been renamed to `seed` and will be removed"
        warnings.warn(msg)
    seed = kwargs.get("seed", random_seed)
    if axis is None:
        func_kwargs = dict()
    else:
        func_kwargs = dict(axis=axis)

    # Initialize the resampler
    if isinstance(seed, np.random.RandomState):
        rng = seed
    else:
        rng = np.random.default_rng(seed)

    # Coerce to arrays
    args = list(map(np.asarray, args))
    if units is not None:
        units = np.asarray(units)

    if isinstance(func, str):

        # Allow named numpy functions
        f = getattr(np, func)

        # Try to use nan-aware version of function if necessary
        missing_data = np.isnan(np.sum(np.column_stack(args)))

        if missing_data and not func.startswith("nan"):
            nanf = getattr(np, f"nan{func}", None)
            if nanf is None:
                msg = f"Data contain nans but no nan-aware version of `{func}` found"
                warnings.warn(msg, UserWarning)
            else:
                f = nanf

    else:
        f = func

    # Handle numpy changes
    try:
        integers = rng.integers
    except AttributeError:
        integers = rng.randint


    boot_dist = []
    for i in range(int(n_boot)):
        resampler = integers(0, n, n, dtype=np.intp)  # intp is indexing dtype
        sample = [a.take(resampler, axis=0) for a in args]
        boot_dist.append(f(*sample, **func_kwargs))
    return np.array(boot_dist)

In [ ]:
filter_callable = admix_div_mean.loc[admix_div_mean.callable_frac > 0.75]
filter_callable.loc[filter_callable['Origin'] == 'Gog Woreda, Gambella region, Ethiopia_eth_case', ['Origin']] = 'Gog Woreda, Ethiopia'
filter_callable["diversity"].quantile([0.005, 0.995])

In [ ]:
len(filter_callable)/len(admix_div_mean)

In [ ]:
filter_callable = admix_div_mean.loc[admix_div_mean.callable_frac > 0.75]
filter_callable.loc[filter_callable['Origin'] == 'Gog Woreda, Gambella region, Ethiopia_eth_case', ['Origin']] = 'Gog Woreda, Ethiopia'
filter_qcut = filter_callable.loc[(filter_callable.diversity >= 0.000513) &
                                 (filter_callable.diversity <= 0.00408)]

In [ ]:
o_l, stat_l, pval_l = [], [], []
for o in result_order:
    o_div = filter_callable.loc[filter_callable.Origin == o]
    o_div = o_div.loc[(o_div.minor_parent_percentage < 0.5)]
    st, pval = stats.spearmanr(o_div.diversity, o_div["minor_parent_percentage"])
    print(o, stats.spearmanr(o_div.diversity, o_div["minor_parent_percentage"]))
    o_l.append(o), stat_l.append(st), pval_l.append(pval)

In [ ]:
pd.DataFrame({"Origin": o_l, "Statistic": stat_l, "P-Value": pval_l})

In [ ]:
import statsmodels.formula.api as smf

o_l, slope_l, intercept_l, rval_l, pval_l, stderr_l = [], [], [], [], [], []
for o in result_order:
    o_div = filter_qcut.loc[filter_qcut.Origin == o]
    glm_results = smf.glm(formula = "minor_parent_percentage ~ diversity", data=o_div).fit()
    print(o, glm_results.summary())

In [ ]:
import statsmodels.api as sm
import numpy as np

df_l = []
for o in result_order:
    o_div = filter_qcut.loc[filter_qcut.Origin == o]
    o_div = o_div.loc[(o_div.callable_frac > 0.75)]
    Y = o_div["minor_parent_percentage"]
    X = o_div.diversity
    X = sm.add_constant(X)
    model = sm.OLS(Y,X)
    results = model.fit()
    print(o)
    #print(results.t_test([1, 0]))
    het_test = sm.stats.het_breuschpagan(resid=results.resid, exog_het=X)
    print(het_test)
    #print(sm.stats.diagnostic.linear_harvey_collier(results))
    df_l.append(list(het_test))
bp_df = pd.DataFrame(df_l, columns=["Lagrange Multiplier", "Lagrange Multiplier P-value", "F-statistic", "F-statistic P-value"])
bp_df["Origin"] = result_order
bp_df

In [ ]:
filter_qcut["Diversity Quantile"] = pd.qcut(filter_qcut.diversity, 5,
                                                       labels=["0-20","20-40",
                                                              "40-60","60-80",
                                                              "80-100"])
g = sns.FacetGrid(filter_qcut.loc[filter_qcut.Origin.isin(result_order_no_gog)], col="Origin",
                  col_wrap = 4, col_order=result_order_no_gog)
g.map_dataframe(sns.pointplot, y="minor_parent_percentage", x="Diversity Quantile", 
                linestyles="none", capsize=.3, errorbar=("ci", 95), scale = 0.6)
#g.map_dataframe(sns.stripplot, x="minor_parent_percentage", y="Diversity Quantile", alpha=0.05)
#g.set(xlim=(0, 0.25))
g.set_titles(col_template="{col_name}")
g.set(xlabel="Diversity Quantile", ylabel="Minor Parent Ancestry")

In [ ]:
o_l, low_l, high_l, effect_l, effect_hl_l = [], [], [], [], []
filter_qcut["Diversity Quantile"] = pd.qcut(filter_qcut.diversity, 5,
                                                       labels=["0-20","20-40",
                                                              "40-60","60-80",
                                                              "80-100"])
for o in result_order:
    s_df = filter_qcut.loc[filter_qcut.Origin == o]
    #print(o, s_df.groupby(["Diversity Quantile"])[["minor_parent_percentage"]].mean())
    low_mean = s_df.loc[s_df["Diversity Quantile"] == "0-20"][["minor_parent_percentage"]].mean()[0]*100
    high_mean = s_df.loc[s_df["Diversity Quantile"] == "80-100"][["minor_parent_percentage"]].mean()[0]*100
    o_l.append(o)
    low_l.append(low_mean)
    high_l.append(high_mean)
    effect_l.append(high_mean/low_mean-1)

In [ ]:
mean_quantile_df = pd.DataFrame({"Origin": o_l, "0-20 Minor Parent Percentage": low_l, 
                        "80-100 Minor Parent Percentage": high_l, "Relative Increase": effect_l})
mean_quantile_df["Absolute Increase"] = mean_quantile_df["80-100 Minor Parent Percentage"]-mean_quantile_df["0-20 Minor Parent Percentage"]
mean_quantile_df

In [ ]:
o_l, low_l, high_l, effect_l, effect_hl_l = [], [], [], [], []
filter_qcut["Diversity Quantile"] = pd.qcut(filter_qcut.diversity, 5,
                                                       labels=["0-20","20-40",
                                                              "40-60","60-80",
                                                              "80-100"])
for o in result_order:
    s_df = filter_qcut.loc[filter_qcut.Origin == o]
    #print(o, s_df.groupby(["Diversity Quantile"])[["minor_parent_percentage"]].mean())
    low_mean = s_df.loc[s_df["Diversity Quantile"] == "0-20"][["minor_parent_percentage"]].mean()[0]*100
    high_mean = s_df.loc[s_df["Diversity Quantile"] == "80-100"][["minor_parent_percentage"]].mean()[0]*100
    o_l.append(o)
    low_l.append(low_mean)
    high_l.append(high_mean)
    effect_l.append(high_mean/low_mean-1)

In [ ]:
filter_qcut

In [ ]:
sns.lmplot(filter_qcut.loc[filter_qcut.Origin == "Gombe, Tanzania"].groupby(["chrom"])[["diversity", "minor_parent_percentage"]].mean(),
               x="diversity", y="minor_parent_percentage")

In [ ]:
o_l, slope_l, intercept_l, pval_slope_l, pval_intercept_l, stderr_slope_l, stderr_intercept_l = [], [], [], [], [], [], []
for o in result_order:
    o_div = filter_callable.loc[filter_callable.Origin == o]
    glm_results = smf.glm(formula = "minor_parent_percentage ~ diversity", data=o_div).fit()
    print(o, glm_results.summary())
    o_l.append(o), slope_l.append(glm_results.params[1]), intercept_l.append(glm_results.params[0])
    pval_slope_l.append(glm_results.pvalues[1]), pval_intercept_l.append(glm_results.pvalues[0])
    stderr_slope_l.append(glm_results.bse[1]), stderr_intercept_l.append(glm_results.bse[0])

In [ ]:
filter_callable.Origin.unique()

In [ ]:
glm_diversity_df = pd.DataFrame({"Origin": o_l, "Intercept": intercept_l, "Slope": slope_l, 
                                 "Intercept P-value": pval_intercept_l, "Slope P-value": pval_slope_l,
                                "Intercept stderr": stderr_intercept_l, "Slope stderr": stderr_slope_l})
glm_diversity_df

In [ ]:
filter_callable["diversity"].quantile([0.05, 0.95])

In [ ]:
o_l, slope_l, intercept_l, pval_slope_l, pval_intercept_l, stderr_slope_l, stderr_intercept_l = [], [], [], [], [], [], []
for o in result_order:
    o_div = filter_callable.loc[(filter_callable.Origin == o) #& (filter_callable.diversity <= 0.0028)
                               #& (filter_callable.diversity >= 0.000819)
                               ]
    glm_results = smf.glm(formula = "minor_parent_percentage ~ diversity",
                          data=o_div, var_weights=np.asarray(o_div["diversity"])
                         ).fit()
    print(o, glm_results.summary())
    o_l.append(o), slope_l.append(glm_results.params[1]), intercept_l.append(glm_results.params[0])
    pval_slope_l.append(glm_results.pvalues[1]), pval_intercept_l.append(glm_results.pvalues[0])
    stderr_slope_l.append(glm_results.bse[1]), stderr_intercept_l.append(glm_results.bse[0])

In [ ]:
glm_diversity_df = pd.DataFrame({"Origin": o_l, "Intercept": intercept_l, "Slope": slope_l, 
                                 "Intercept P-value": pval_intercept_l, "Slope P-value": pval_slope_l,
                                "Intercept stderr": stderr_intercept_l, "Slope stderr": stderr_slope_l})
glm_diversity_df

In [ ]:
o_l, slope_l, intercept_l, pval_slope_l, pval_intercept_l, stderr_slope_l, stderr_intercept_l = [], [], [], [], [], [], []
for o in result_order:
    o_div = filter_qcut.loc[filter_qcut.Origin == o]
    glm_results = smf.glm(formula = "minor_parent_percentage ~ diversity", data=o_div,
                         var_weights=np.asarray(o_div["diversity"])).fit()
    print(o, glm_results.summary())
    o_l.append(o), slope_l.append(glm_results.params[1]), intercept_l.append(glm_results.params[0])
    pval_slope_l.append(glm_results.pvalues[1]), pval_intercept_l.append(glm_results.pvalues[0])
    stderr_slope_l.append(glm_results.bse[1]), stderr_intercept_l.append(glm_results.bse[0])

In [ ]:
glm_diversity_df = pd.DataFrame({"Origin": o_l, "Intercept": intercept_l, "Slope": slope_l, 
                                 "Intercept P-value": pval_intercept_l, "Slope P-value": pval_slope_l,
                                "Intercept stderr": stderr_intercept_l, "Slope stderr": stderr_slope_l})
glm_diversity_df

In [ ]:
groups = filter_qcut.groupby("chrom")
aut_mean, chrom_mean = groups["diversity"].transform("mean"), filter_qcut["diversity"].mean()
filter_qcut["aut_norm_div"] = filter_qcut.diversity*chrom_mean/aut_mean

In [ ]:
o_l, slope_l, intercept_l, pval_slope_l, pval_intercept_l, stderr_slope_l, stderr_intercept_l = [], [], [], [], [], [], []
for o in result_order:
    o_div = filter_qcut.loc[filter_qcut.Origin == o]
    glm_results = smf.glm(formula = "minor_parent_percentage ~ aut_norm_div", data=o_div,
                         var_weights=np.asarray(o_div["aut_norm_div"])).fit()
    print(o, glm_results.summary())
    o_l.append(o), slope_l.append(glm_results.params[1]), intercept_l.append(glm_results.params[0])
    pval_slope_l.append(glm_results.pvalues[1]), pval_intercept_l.append(glm_results.pvalues[0])
    stderr_slope_l.append(glm_results.bse[1]), stderr_intercept_l.append(glm_results.bse[0])

In [ ]:
glm_diversity_df = pd.DataFrame({"Origin": o_l, "Intercept": intercept_l, "Slope": slope_l, 
                                 "Intercept P-value": pval_intercept_l, "Slope P-value": pval_slope_l,
                                "Intercept stderr": stderr_intercept_l, "Slope stderr": stderr_slope_l})
glm_diversity_df

In [ ]:
o_div = filter_qcut.loc[filter_qcut.Origin.isin(["Mikumi, Tanzania", "Ruaha, Tanzania"])]
glm_results = smf.glm(formula = "minor_parent_percentage ~ diversity*Origin", data=o_div,
                         var_weights=np.asarray(o_div["diversity"])).fit()
print(glm_results.summary(), glm_results.pvalues)

In [ ]:
o_div = filter_qcut.loc[filter_qcut.Origin.isin(["Ngorongoro, Tanzania", "Tarangire, Tanzania"])]
glm_results = smf.glm(formula = "minor_parent_percentage ~ diversity*Origin", data=o_div,
                         var_weights=np.asarray(o_div["diversity"])).fit()
print(glm_results.summary(), glm_results.pvalues)

In [ ]:
from scipy.stats import norm

predictions = glm_results.get_prediction()
df_predictions = predictions.summary_frame()
df_predictions["diversity"] = o_div["diversity"]

In [ ]:
g = sns.histplot(filter_qcut.loc[filter_qcut.Origin.isin(gog)], x="diversity")
g.set(xlabel = "Background Diversity")
g.set(xticks=np.arange(0.0005, 0.004, 0.001))

In [ ]:
g = sns.histplot(filter_callable.loc[filter_callable.Origin.isin(gog)], x="diversity")
g.set(xlabel = "Background Diversity")
#g.set(xticks=np.arange(0.0005, 0.014, 0.001))

In [ ]:
filter_qcut.loc[filter_qcut.Origin.isin(gog)]["diversity"].mean(), filter_qcut.loc[filter_qcut.Origin.isin(gog)]["diversity"].median()

In [ ]:
g = lmplot(filter_qcut.loc[filter_qcut.Origin.isin(yellows)], x="diversity", y="minor_parent_percentage",
                            scatter=False, hue="Origin", weighted=True, n_boot=1000, ci=99.5) #Additional bootstraps makes it take long
g.set(ylabel="Minor Parent Ancestry Proportion", xlabel="Background Diversity", title="Tanzanian Yellow Baboons")
g.set(ylim=(-0.01, None))
g.set(xticks=np.arange(0.0005, 0.004, 0.001))

In [ ]:
g = lmplot(filter_qcut.loc[filter_qcut.Origin.isin(olives)], x="diversity", y="minor_parent_percentage",
                            scatter=False, hue="Origin", weighted=True, n_boot=1000, ci=99.5) #Additional bootstraps makes it take long
g.set(ylabel="Minor Parent Ancestry Proportion", xlabel="Background Diversity", title="Tanzanian Olive Baboons")
g.set(ylim=(-0.01, 0.11))
g.set(xticks=np.arange(0.0005, 0.004, 0.001))

In [ ]:
g = lmplot(filter_qcut.loc[filter_qcut.Origin.isin(olives_with_tarangire)], x="diversity", y="minor_parent_percentage",
                            scatter=False, hue="Origin", weighted=True, n_boot=1000, ci=99.5) #Additional bootstraps makes it take long
g.set(ylabel="Minor Parent Ancestry Proportion", xlabel="Background Diversity", title="Tanzanian Olive Baboons")
g.set(ylim=(-0.01, None))
g.set(xticks=np.arange(0.0005, 0.004, 0.001))

In [ ]:
g = lmplot(filter_qcut.loc[filter_qcut.Origin.isin(gog)], x="diversity", y="minor_parent_percentage",
                            scatter=False, weighted=True, n_boot=1000, ci=99.5) #Additional bootstraps makes it take long
g.set(ylabel="Minor Parent Ancestry Proportion", xlabel="Background Diversity", title="Gog Olive Baboons")
g.set(ylim=(-0.01, None))
g.set(xticks=np.arange(0.0005, 0.004, 0.001))

In [ ]:
g = lmplot(filter_callable.loc[filter_callable.Origin.isin(["Mahale, Tanzania",
                                                           "Katavi, Tanzania",
                                                           "Issa Valley, Tanzania"])], x="diversity", y="minor_parent_percentage",
                            scatter=False, hue="Origin", weighted=True, n_boot=1000, ci=99.5) #Additional bootstraps makes it take long
g.set(ylabel="Minor Parent Ancestry Proportion", xlabel="Background Diversity", title="Tanzanian Kinda-like Baboons")
g.set(ylim=(-0.01, None))

In [ ]:
g = lmplot(filter_callable.loc[filter_callable.Origin.isin(olives_with_tarangire)], x="diversity", y="minor_parent_percentage",
                            scatter=False, hue="Origin", weighted=True, n_boot=1000, ci=99.5) #Additional bootstraps makes it take long
g.set(ylabel="Minor Parent Ancestry Proportion", xlabel="Background Diversity", title="Tanzanian Olive Baboons")
g.set(ylim=(-0.01, None))

In [ ]:
o_l, stat_l, pval_l = [], [], []
for o in result_order:
    o_div = filter_qcut.loc[filter_qcut.Origin == o]
    #o_div = o_div.loc[(o_div.minor_parent_percentage < 0.5)]
    st, pval = stats.pearsonr(o_div.diversity, o_div["minor_parent_percentage"])
    print(o, stats.pearsonr(o_div.diversity, o_div["minor_parent_percentage"]))
    o_l.append(o), stat_l.append(st), pval_l.append(pval)
pd.DataFrame({"Origin": o_l, "Statistic": stat_l, "P-Value": pval_l})

Recomb investigation for autosomes

In [ ]:
filter_callable = admix_div_mean.loc[admix_div_mean.callable_frac > 0.75]
filter_callable.loc[filter_callable['Origin'] == 'Gog Woreda, Gambella region, Ethiopia_eth_case', ['Origin']] = 'Gog Woreda, Ethiopia'
filter_callable["average_cM_window"].quantile([0.005, 0.995])

In [ ]:
len(filter_callable)

In [ ]:
len(filter_callable)/len(admix_div_mean)

In [ ]:
filter_qcut_recomb = filter_callable.loc[(filter_callable.average_cM_window >= 1.13e-07) &
                                 (filter_callable.average_cM_window <=  4.08e-06)]
filter_callable["window_cM"] = filter_callable["average_cM_window"]*100000
filter_qcut_recomb["window_cM"] = filter_qcut_recomb["average_cM_window"]*100000

In [ ]:
import statsmodels.api as sm
import numpy as np

df_l = []
for o in result_order:
    o_div = filter_qcut_recomb.loc[filter_qcut_recomb.Origin == o]
    Y = o_div["minor_parent_percentage"]
    X = o_div.window_cM
    X = sm.add_constant(X)
    model = sm.OLS(Y,X)
    results = model.fit()
    print(o)
    #print(results.t_test([1, 0]))
    het_test = sm.stats.het_breuschpagan(resid=results.resid, exog_het=X)
    print(het_test)
    #print(sm.stats.diagnostic.linear_harvey_collier(results))
    df_l.append(list(het_test))
bp_df = pd.DataFrame(df_l, columns=["Lagrange Multiplier", "Lagrange Multiplier P-value", "F-statistic", "F-statistic P-value"])
bp_df["Origin"] = result_order
bp_df

In [ ]:
filter_qcut_recomb["Recombination Quantile"] = pd.qcut(filter_qcut_recomb.window_cM, 5,
                                                       labels=["0-20","20-40",
                                                              "40-60","60-80",
                                                              "80-100"])
g = sns.FacetGrid(filter_qcut_recomb.loc[filter_qcut_recomb.Origin.isin(result_order_no_gog)], col="Origin",
                  col_wrap = 4, col_order=result_order_no_gog)
g.map_dataframe(sns.pointplot, y="minor_parent_percentage", x="Recombination Quantile", 
                linestyles="none", capsize=.3, errorbar=("ci", 95), scale = 0.6)
#g.map_dataframe(sns.stripplot, x="minor_parent_percentage", y="Diversity Quantile", alpha=0.05)
#g.set(xlim=(0, 0.25))
g.set_titles(col_template="{col_name}")
g.set(xlabel="Recombination Quantile", ylabel="Minor Parent Ancestry")

In [ ]:
o_l, low_l, high_l, effect_l, effect_hl_l = [], [], [], [], []
filter_qcut_recomb["Recombination Quantile"] = pd.qcut(filter_qcut_recomb.window_cM, 5,
                                                       labels=["0-20","20-40",
                                                              "40-60","60-80",
                                                              "80-100"])
for o in result_order:
    s_df = filter_qcut_recomb.loc[filter_qcut_recomb.Origin == o]
    #print(o, s_df.groupby(["Recombination Quantile"])[["minor_parent_percentage"]].mean())
    low_mean = s_df.loc[s_df["Recombination Quantile"] == "0-20"][["minor_parent_percentage"]].mean()[0]*100
    high_mean = s_df.loc[s_df["Recombination Quantile"] == "80-100"][["minor_parent_percentage"]].mean()[0]*100
    o_l.append(o)
    low_l.append(low_mean)
    high_l.append(high_mean)
    effect_l.append(high_mean/low_mean-1)

In [ ]:
mean_quantile_df = pd.DataFrame({"Origin": o_l, "0-20 Minor Parent Percentage": low_l, 
                        "80-100 Minor Parent Percentage": high_l, "Relative Increase": effect_l})
mean_quantile_df["Absolute Increase"] = mean_quantile_df["80-100 Minor Parent Percentage"]-mean_quantile_df["0-20 Minor Parent Percentage"]
mean_quantile_df

In [ ]:
o_l, slope_l, intercept_l, pval_slope_l, pval_intercept_l, stderr_slope_l, stderr_intercept_l = [], [], [], [], [], [], []
for o in result_order:
    o_div = filter_callable.loc[filter_callable.Origin == o]
    glm_results = smf.glm(formula = "minor_parent_percentage ~ window_cM", data=o_div,
                         var_weights=np.asarray(o_div["window_cM"])).fit()
    print(o, glm_results.summary())
    o_l.append(o), slope_l.append(glm_results.params[1]), intercept_l.append(glm_results.params[0])
    pval_slope_l.append(glm_results.pvalues[1]), pval_intercept_l.append(glm_results.pvalues[0])
    stderr_slope_l.append(glm_results.bse[1]), stderr_intercept_l.append(glm_results.bse[0])

In [ ]:
glm_recomb_df = pd.DataFrame({"Origin": o_l, "Intercept": intercept_l, "Slope": slope_l, 
                                 "Intercept P-value": pval_intercept_l, "Slope P-value": pval_slope_l,
                                "Intercept stderr": stderr_intercept_l, "Slope stderr": stderr_slope_l})
glm_recomb_df

In [ ]:
o_l, slope_l, intercept_l, pval_slope_l, pval_intercept_l, stderr_slope_l, stderr_intercept_l = [], [], [], [], [], [], []
for o in result_order:
    o_div = filter_qcut_recomb.loc[filter_qcut_recomb.Origin == o]
    glm_results = smf.glm(formula = "minor_parent_percentage ~ window_cM", data=o_div,
                         var_weights=np.asarray(o_div["window_cM"])
                         ).fit()
    print(o, glm_results.summary())
    o_l.append(o), slope_l.append(glm_results.params[1]), intercept_l.append(glm_results.params[0])
    pval_slope_l.append(glm_results.pvalues[1]), pval_intercept_l.append(glm_results.pvalues[0])
    stderr_slope_l.append(glm_results.bse[1]), stderr_intercept_l.append(glm_results.bse[0])

In [ ]:
glm_recomb_df = pd.DataFrame({"Origin": o_l, "Intercept": intercept_l, "Slope": slope_l, 
                                 "Intercept P-value": pval_intercept_l, "Slope P-value": pval_slope_l,
                                "Intercept stderr": stderr_intercept_l, "Slope stderr": stderr_slope_l})
glm_recomb_df

In [ ]:
filter_callable.loc[filter_callable['Origin'] == 'Gog Woreda, Gambella region, Ethiopia_eth_case', ['Origin']] = 'Gog Woreda, Ethiopia'
filter_callable["average_cM_window"].quantile([0.000, 0.99])

In [ ]:
g = sns.histplot(filter_qcut_recomb.loc[filter_qcut_recomb.Origin.isin(gog)], x="window_cM")
g.set(xlabel="Recombination across window (cM)")

In [ ]:
filter_qcut_recomb.loc[filter_qcut_recomb.Origin.isin(gog)]["window_cM"].mean(), filter_qcut_recomb.loc[filter_qcut_recomb.Origin.isin(gog)]["window_cM"].median()

In [ ]:
g = sns.histplot(filter_callable.loc[filter_callable.Origin.isin(gog)], x="window_cM")
g.set(xlabel="Recombination across window (cM)")

In [ ]:
filter_callable.loc[filter_callable.Origin.isin(gog)]["window_cM"].mean(), filter_callable.loc[filter_callable.Origin.isin(gog)]["window_cM"].median()

In [ ]:
g = lmplot(filter_qcut_recomb.loc[filter_qcut_recomb.Origin.isin(yellows)], x="window_cM", y="minor_parent_percentage",
                            scatter=False, hue="Origin", weighted=True, n_boot=1000, ci=99.5)
g.set(ylabel="Minor Parent Ancestry Proportion", xlabel="Recombination across window (cM)", title="Tanzanian Yellow Baboons")
g.set(ylim=(-0.01, None))

In [ ]:
g = lmplot(filter_qcut_recomb.loc[filter_qcut_recomb.Origin.isin(olives)], x="window_cM", y="minor_parent_percentage",
                            scatter=False, hue="Origin", weighted=True, n_boot=1000, ci=95)
g.set(ylabel="Minor Parent Ancestry Proportion", xlabel="Recombination across window (cM)", title="Tanzanian Olive Baboons")
g.set(ylim=(-0.01, 0.11))

In [ ]:
g = lmplot(filter_qcut_recomb.loc[filter_qcut_recomb.Origin.isin(olives_with_tarangire)], x="window_cM", y="minor_parent_percentage",
                            scatter=False, hue="Origin", weighted=True, n_boot=1000, ci=95)
g.set(ylabel="Minor Parent Ancestry Proportion", xlabel="Recombination across window (cM)", title="Tanzanian Olive Baboons")
#g.set(ylim=(-0.01, 0.11))

In [ ]:
g = lmplot(filter_callable.loc[filter_callable.Origin.isin(yellows)], x="window_cM", y="minor_parent_percentage",
                            scatter=False, hue="Origin", weighted=True, n_boot=1000, ci=99.5)
g.set(ylabel="Minor Parent Ancestry Proportion", xlabel="Recombination across window (cM)", title="Tanzanian Yellow Baboons")
g.set(ylim=(-0.01, None))

In [ ]:
g = lmplot(filter_callable.loc[filter_callable.Origin.isin(olives_with_tarangire)], x="window_cM", y="minor_parent_percentage",
                            scatter=False, hue="Origin", weighted=True, n_boot=1000, ci=99.5)
g.set(ylabel="Minor Parent Ancestry Proportion", xlabel="Recombination across window (cM)", title="Tanzanian Olive Baboons")
g.set(ylim=(-0.01, None))

In [ ]:
g = lmplot(filter_qcut_recomb.loc[filter_qcut_recomb.Origin.isin(gog)], x="window_cM", y="minor_parent_percentage",
                            scatter=False, weighted=True, n_boot=1000, ci=99.5)
g.set(ylabel="Minor Parent Ancestry Proportion", xlabel="Recombination across window (cM)", title="Gog Olive Baboon")
g.set(ylim=(-0.01, None))

In [ ]:
norm_filter = filter_qcut_recomb.loc[(filter_qcut_recomb.diversity >= 0.000513) &
                                 (filter_qcut_recomb.diversity <= 0.00408)]
g = sns.regplot(norm_filter.loc[(norm_filter.Origin == "Serengeti, Tanzania") &
                              #(admix_div_mean["minor_parent_percentage"] < 0.5) &
                              (norm_filter.callable_frac > 0.75)], x="window_cM", y="diversity",
                            ci=95, scatter_kws={'alpha':0.3})
g.set(xlabel="Recombination across window (cM)", ylabel = "Background Diversity")

In [ ]:
stats.pearsonr(norm_filter.window_cM, norm_filter.diversity)

In [ ]:
norm_filter["norm_diversity"] = (norm_filter.diversity-norm_filter.diversity.mean())/(norm_filter.diversity.std())
norm_filter["norm_recomb"] = (norm_filter.window_cM-norm_filter.window_cM.mean())/(norm_filter.window_cM.std())

In [ ]:
o_l, cd_l, cr_l, nd_l, rd_l = [], [], [], [], []
for o in result_order:
    o_div = norm_filter.loc[norm_filter.Origin == o]
    glm_results = smf.glm(formula = "minor_parent_percentage ~ norm_diversity", data=o_div
                         ).fit()
    print(o, glm_results.summary2(), glm_results.pvalues, glm_results.f_test(np.identity(len(glm_results.params))[1:,:]))
    #o_l.append(o), cd_l.append(glm_results.params[1]), cr_l.append(glm_results.params[2])
    #nd_l.append(glm_results.pvalues[1]), rd_l.append(glm_results.pvalues[2])

In [ ]:
o_l, cd_l, cr_l, nd_l, rd_l = [], [], [], [], []
for o in result_order:
    o_div = norm_filter.loc[norm_filter.Origin == o]
    glm_results = smf.glm(formula = "minor_parent_percentage ~ diversity + window_cM", data=o_div
                         ).fit()
    print(o, glm_results.summary2(), glm_results.pvalues, glm_results.f_test(np.identity(len(glm_results.params))[1:,:]))
    o_l.append(o), cd_l.append(glm_results.params[1]), cr_l.append(glm_results.params[2])
    nd_l.append(glm_results.pvalues[1]), rd_l.append(glm_results.pvalues[2])

In [ ]:
pd.DataFrame({"Origin": o_l, "Recombination Slope": cr_l, "Diversity Slope": cd_l,
              "Recombination p-val": rd_l, "Diversity p-val": nd_l})

In [ ]:
o_l, cd_l, cr_l, nd_l, rd_l, i_l, id_l = [], [], [], [], [], [], []
for o in result_order:
    o_div = norm_filter.loc[norm_filter.Origin == o]
    glm_results = smf.glm(formula = "minor_parent_percentage ~ norm_diversity * norm_recomb", data=o_div
                         ).fit()
    print(o, glm_results.summary2(), glm_results.pvalues, glm_results.f_test(np.identity(len(glm_results.params))[1:,:]))
    o_l.append(o), cd_l.append(glm_results.params[1]), cr_l.append(glm_results.params[2])
    nd_l.append(glm_results.pvalues[1]), rd_l.append(glm_results.pvalues[2])
    i_l.append(glm_results.params[3]), id_l.append(glm_results.pvalues[3])

In [ ]:
pd.DataFrame({"Origin": o_l, "Recombination Slope": cr_l, "Diversity Slope": cd_l, "Interaction Slope": i_l,
              "Recombination p-val": rd_l, "Diversity p-val": nd_l, "Interaction p-val": id_l})

In [ ]:
o_l, slope_l, intercept_l, pval_slope_l, pval_intercept_l, stderr_slope_l, stderr_intercept_l = [], [], [], [], [], [], []
for o in result_order:
    o_div = norm_filter.loc[norm_filter.Origin == o]
    glm_results = smf.ols(formula = "minor_parent_percentage ~ norm_diversity", data=o_div
                         ).fit()
    print(o, glm_results.summary2())

In [ ]:
g = sns.jointplot(norm_filter.loc[(norm_filter.Origin == "Serengeti, Tanzania") &
                              #(admix_div_mean["minor_parent_percentage"] < 0.5) &
                              (norm_filter.callable_frac > 0.9)], x="norm_diversity", y="norm_recomb")
g.set_axis_labels(xlabel="Normlized Diversity", ylabel="Normalized Recombination in cM")

In [ ]:
ser_div = admix_div_mean.loc[admix_div_mean.Origin == "Serengeti, Tanzania"].reset_index()
ser_div["tarangire_diff"] = (admix_div_mean.loc[admix_div_mean.Origin == "Serengeti, Tanzania"].minor_parent_percentage.reset_index()-admix_div_mean.loc[admix_div_mean.Origin == "Tarangire, Tanzania"].minor_parent_percentage.reset_index()).minor_parent_percentage
ser_div = ser_div.loc[(ser_div.callable_frac > 0.9)]
glm_results = smf.glm(formula = "tarangire_diff ~ diversity", data=ser_div,
                         var_weights=np.asarray(ser_div.diversity)
                         ).fit()
print(glm_results.summary(), glm_results.pvalues[1])

In [ ]:
yellow_pops = filter_qcut_recomb.loc[filter_qcut_recomb.Origin.isin(["Mahale, Tanzania", "Ruaha, Tanzania"])]
glm_results = smf.glm(formula = "minor_parent_percentage ~ diversity*Origin", data=yellow_pops,
                         var_weights=np.asarray(yellow_pops.diversity)
                         ).fit()
print(glm_results.summary(), glm_results.pvalues[0], glm_results.pvalues[3])

ChrX case

In [ ]:
chrX_mean = mean_window_df_tanz_eth.loc[mean_window_df_tanz_eth.chrom == "all_chrX"]
chrX_mean["chrom"] = "chrX"
admix_div_chrX_all = mean_diversity.merge(chrX_mean, on=["chrom", "start"])
admix_div_chrX_all["North Percentage"] = admix_div_chrX_all.north_sum/100000
admix_div_chrX_all = admix_div_chrX_all.merge(c_r_g_df, on=["chrom", "start"])
admix_div_chrX_all["Species"] = admix_div_chrX_all.Origin.map(dict(zip(meta_data_samples_sci.Origin, meta_data_samples_sci.Species)))


In [ ]:
#Selecting the cases of interest and setting Minor Parent Ancestry

admix_div_chrX_all = admix_div_chrX_all.loc[admix_div_chrX_all.Origin.isin(origins_interest)]
admix_div_chrX_all["minor_parent_percentage"] = [x if y == ("cynocephalus") or z == "Gog Woreda, Gambella region, Ethiopia_eth_case"
                                             else 1-x for x, y, z in zip(admix_div_chrX_all["North Percentage"],
                                                                         admix_div_chrX_all["Species"],
                                                                        admix_div_chrX_all["Origin"])]
admix_div_chrX_all["local_minor_ancestry"] = [min(x, 1-x) for x in admix_div_chrX_all["North Percentage"]]
filter_callable_chrX = admix_div_chrX_all.loc[admix_div_chrX_all.callable_frac > 0.9]
filter_callable_chrX.loc[filter_callable_chrX['Origin'] == 'Gog Woreda, Gambella region, Ethiopia_eth_case', ['Origin']] = 'Gog Woreda, Ethiopia'
filter_callable_chrX["diversity"].quantile([0.005, 0.995])

Plot of difference in frequencies across the autosome and ChrX.

In [ ]:
filter_callable_all = pd.concat([filter_callable_chrX, filter_callable])
filter_callable_all["chrom_type"] = ["autosome" if x != "chrX" else "chrX" for x in filter_callable_all.chrom]

In [ ]:
g = sns.FacetGrid(filter_callable_all.loc[filter_callable_all.Origin.isin(["Gog Woreda, Ethiopia"])])
g.map_dataframe(sns.histplot, x="minor_parent_percentage", hue="chrom_type", hue_order=["autosome", "chrX"],
                 common_norm = False, bins=10, stat="percent") 
g.add_legend()
g.set(yscale="log")
#g.set_titles(col_template="{col_name}")
g.set(xlabel="Minor Parent Ancestry")

In [ ]:
g = sns.FacetGrid(filter_callable_all.loc[filter_callable_all.Origin.isin(result_order_no_gog)], col="Origin",
                  col_wrap = 4, col_order=result_order_no_gog,legend_out=True)
g.map_dataframe(sns.histplot, x="minor_parent_percentage", hue="chrom_type", hue_order=["autosome", "chrX"],
                 common_norm = False, bins=10, stat="percent") 
g.add_legend()
g.set(yscale="log")
g.set_titles(col_template="{col_name}")
g.set(xlabel="Minor Parent Ancestry")


In [ ]:
g = sns.FacetGrid(filter_callable_all.loc[filter_callable_all.Origin.isin(result_order_no_gog)], col="Origin",
                  col_wrap = 4, col_order=result_order_no_gog,legend_out=True)
g.map_dataframe(sns.histplot, x="minor_parent_percentage", hue="chrom_type", hue_order=["autosome", "chrX"],
                 common_norm = False, bins=10, stat="percent") 
g.add_legend()
g.set(ylim=(0, 5))
g.set_titles(col_template="{col_name}")
g.set(xlabel="Minor Parent Ancestry")

In [ ]:
result_order_no_gog_tarangire = ['Selous, Tanzania', 'Udzungwa, Tanzania', 'Mahale, Tanzania', 'Katavi, Tanzania', 'Ruaha, Tanzania',
      'Arusha, Tanzania', 'Ngorongoro, Tanzania', 'Gombe, Tanzania', 
             'Lake Manyara, Tanzania', 'Serengeti, Tanzania']

In [ ]:
g = sns.FacetGrid(filter_callable_all.loc[filter_callable_all.Origin.isin(result_order_no_gog_tarangire)].sort_values(by=["chrom"]), col="Origin",
                  col_wrap = 4, col_order=result_order_no_gog_tarangire)
g.map_dataframe(sns.barplot, y="minor_parent_percentage", x="chrom_type", palette=sns.color_palette(), errorbar=None) 
g.set_titles(col_template="{col_name}")
g.set(xlabel="Chromosome Type", ylabel="MPA Proportion")

In [ ]:
filter_callable_all.loc[filter_callable_all.Origin.isin(gog)].chrom.unique()

In [ ]:
filter_qcut_chrX = filter_callable_chrX.loc[(filter_callable_chrX.diversity >= 0.000147) & 
                                           (filter_callable_chrX.diversity <= 0.00156)]

In [ ]:
filter_qcut.loc[filter_qcut.Origin.isin(gog)].diversity.mean(), filter_qcut_chrX.loc[filter_qcut_chrX.Origin.isin(gog)].diversity.mean()

In [ ]:
filter_qcut_chrX.loc[filter_qcut_chrX.Origin.isin(gog)].diversity.mean()/filter_qcut.loc[filter_qcut.Origin.isin(gog)].diversity.mean()

In [ ]:
from scipy.stats import mannwhitneyu
res = mannwhitneyu(filter_qcut.loc[filter_qcut.Origin.isin(gog)].diversity*0.75,
                   filter_qcut_chrX.loc[filter_qcut_chrX.Origin.isin(gog)].diversity, alternative="greater")
print(res)

In [ ]:
import statsmodels.api as sm
df_l = []
for o in result_order:
    o_div = filter_qcut_chrX.loc[filter_qcut_chrX.Origin == o]
    Y = o_div["minor_parent_percentage"]
    X = o_div.diversity
    X = sm.add_constant(X)
    model = sm.OLS(Y,X)
    results = model.fit()
    print(o)
    #print(results.t_test([1, 0]))
    het_test = sm.stats.het_breuschpagan(resid=results.resid, exog_het=X)
    print(het_test)
    #print(sm.stats.diagnostic.linear_harvey_collier(results))
    df_l.append(list(het_test))
bp_df = pd.DataFrame(df_l, columns=["Lagrange Multiplier", "Lagrange Multiplier P-value", "F-statistic", "F-statistic P-value"])
bp_df["Origin"] = result_order
bp_df

In [ ]:
filter_qcut_chrX["Diversity Quantile"] = pd.qcut(filter_qcut_chrX.diversity, 5,
                                                       labels=["0-20","20-40",
                                                              "40-60","60-80",
                                                              "80-100"])
g = sns.FacetGrid(filter_qcut_chrX.loc[filter_qcut_chrX.Origin.isin(result_order_no_gog)], col="Origin",
                  col_wrap = 4, col_order=result_order_no_gog)
g.map_dataframe(sns.pointplot, y="minor_parent_percentage", x="Diversity Quantile", 
                linestyles="none", capsize=.3, errorbar=("ci", 95), scale = 0.6)
#g.map_dataframe(sns.stripplot, x="minor_parent_percentage", y="Diversity Quantile", alpha=0.05)
#g.set(xlim=(0, 0.25))
g.set_titles(col_template="{col_name}")
g.set(xlabel="Diversity Quantile", ylabel="Minor Parent Ancestry")

In [ ]:
filter_qcut["Diversity Quantile"] = pd.qcut(filter_qcut.diversity, 5,
                                                       labels=["0-20","20-40",
                                                              "40-60","60-80",
                                                              "80-100"])
filter_qcut["chrom_type"] = "autosome"

filter_qcut_chrX["Diversity Quantile"] = pd.qcut(filter_qcut_chrX.diversity, 5,
                                                       labels=["0-20","20-40",
                                                              "40-60","60-80",
                                                              "80-100"])
filter_qcut_chrX["chrom_type"] = "chrX"
concat_df = pd.concat([filter_qcut, filter_qcut_chrX])

g = sns.FacetGrid(concat_df.loc[concat_df.Origin.isin(["Gog Woreda, Ethiopia"])], hue="chrom_type")
g.map_dataframe(sns.pointplot, y="minor_parent_percentage", x="Diversity Quantile", 
                linestyles="none", capsize=.3, errorbar=("ci", 95), scale = 0.6)
#g.map_dataframe(sns.stripplot, x="minor_parent_percentage", y="Diversity Quantile", alpha=0.05)
#g.set(xlim=(0, 0.25))
g.set_titles(col_template="{col_name}")
g.set(xlabel="Diversity Quantile", ylabel="Minor Parent Ancestry")
g.fig.set_figwidth(3.8)
g.fig.set_figheight(3.8)

In [ ]:
o_l, low_l, high_l, effect_l, effect_hl_l = [], [], [], [], []
filter_qcut_chrX["Recombination Quantile"] = pd.qcut(filter_qcut_chrX.diversity, 5,
                                                       labels=["0-20","20-40",
                                                              "40-60","60-80",
                                                              "80-100"])
for o in result_order:
    s_df = filter_qcut_chrX.loc[filter_qcut_chrX.Origin == o]
    #print(o, s_df.groupby(["Recombination Quantile"])[["minor_parent_percentage"]].mean())
    low_mean = s_df.loc[s_df["Recombination Quantile"] == "0-20"][["minor_parent_percentage"]].mean()[0]*100
    high_mean = s_df.loc[s_df["Recombination Quantile"] == "80-100"][["minor_parent_percentage"]].mean()[0]*100
    o_l.append(o)
    low_l.append(low_mean)
    high_l.append(high_mean)
    effect_l.append(high_mean/low_mean-1)

mean_quantile_df = pd.DataFrame({"Origin": o_l, "0-20 Minor Parent Percentage": low_l, 
                        "80-100 Minor Parent Percentage": high_l, "Relative Increase": effect_l})
mean_quantile_df["Absolute Increase"] = mean_quantile_df["80-100 Minor Parent Percentage"]-mean_quantile_df["0-20 Minor Parent Percentage"]
mean_quantile_df

In [ ]:
o_l, low_l, high_l, effect_l, effect_hl_l = [], [], [], [], []
filter_qcut_chrX["Diversity Quantile"] = pd.qcut(filter_qcut_chrX.diversity, 5,
                                                       labels=["0-20","20-40",
                                                              "40-60","60-80",
                                                              "80-100"])
for o in result_order:
    s_df = filter_qcut_chrX.loc[filter_qcut_chrX.Origin == o]
    #print(o, s_df.groupby(["Diversity Quantile"])[["minor_parent_percentage"]].mean())
    low_mean = s_df.loc[s_df["Diversity Quantile"] == "0-20"][["minor_parent_percentage"]].mean()[0]*100
    high_mean = s_df.loc[s_df["Diversity Quantile"] == "80-100"][["minor_parent_percentage"]].mean()[0]*100
    o_l.append(o)
    low_l.append(low_mean)
    high_l.append(high_mean)
    effect_l.append(high_mean/low_mean)

In [ ]:
import statsmodels.formula.api as smf
import numpy as np
import pandas as pd

In [ ]:
o_l, slope_l, intercept_l, pval_slope_l, pval_intercept_l, stderr_slope_l, stderr_intercept_l = [], [], [], [], [], [], []
for o in result_order:
    o_div = filter_qcut_chrX.loc[filter_qcut_chrX.Origin == o]
    glm_results = smf.glm(formula = "minor_parent_percentage ~ diversity", data=o_div,
                         var_weights=np.asarray(o_div["diversity"])).fit()
    print(o, glm_results.summary())
    o_l.append(o), slope_l.append(glm_results.params[1]), intercept_l.append(glm_results.params[0])
    pval_slope_l.append(glm_results.pvalues[1]), pval_intercept_l.append(glm_results.pvalues[0])
    stderr_slope_l.append(glm_results.bse[1]), stderr_intercept_l.append(glm_results.bse[0])

In [ ]:
o_l, slope_l, intercept_l, pval_slope_l, pval_intercept_l, stderr_slope_l, stderr_intercept_l = [], [], [], [], [], [], []
for o in result_order:
    o_div = filter_qcut_chrX.loc[filter_qcut_chrX.Origin == o]
    glm_results = smf.glm(formula = "minor_parent_percentage ~ diversity", data=o_div,
                         var_weights=np.asarray(o_div["diversity"])).fit()
    print(o, glm_results.summary())
    o_l.append(o), slope_l.append(glm_results.params[1]), intercept_l.append(glm_results.params[0])
    pval_slope_l.append(glm_results.pvalues[1]), pval_intercept_l.append(glm_results.pvalues[0])
    stderr_slope_l.append(glm_results.bse[1]), stderr_intercept_l.append(glm_results.bse[0])

In [ ]:
glm_diversity_df = pd.DataFrame({"Origin": o_l, "Intercept": intercept_l, "Slope": slope_l, 
                                 "Intercept P-value": pval_intercept_l, "Slope P-value": pval_slope_l,
                                "Intercept stderr": stderr_intercept_l, "Slope stderr": stderr_slope_l})
glm_diversity_df

In [ ]:
import matplotlib.ticker as ticker
g = sns.histplot(filter_qcut_chrX.loc[filter_qcut_chrX.Origin.isin(gog)], x="diversity", )
g.set(xlabel = "Background Diversity")
g.xaxis.set_major_locator(ticker.MultipleLocator(0.00025))

In [ ]:
filter_qcut_chrX.loc[filter_qcut_chrX.Origin.isin(gog)].diversity.mean(), filter_qcut_chrX.loc[filter_qcut_chrX.Origin.isin(gog)].diversity.median()

In [ ]:
g = lmplot(filter_qcut_chrX.loc[filter_qcut_chrX.Origin.isin(yellows)], x="diversity", y="minor_parent_percentage",
                            scatter=False, hue="Origin", weighted=True, n_boot=1000, ci=99.5) #Additional bootstraps makes it take long
g.set(ylabel="Minor Parent Ancestry Proportion", xlabel="Background Diversity", title="Tanzanian Yellow Baboons")
g.set(ylim=(-0.01, None))
g.set(xticks=[0, 0.0005, 0.001, 0.0015])

In [ ]:
g = lmplot(filter_qcut_chrX.loc[filter_qcut_chrX.Origin.isin(olives)], x="diversity", y="minor_parent_percentage",
                            scatter=False, hue="Origin", weighted=True, n_boot=1000, ci=99.5) #Additional bootstraps makes it take long
g.set(ylabel="Minor Parent Ancestry Proportion", xlabel="Background Diversity", title="Tanzanian Olive Baboons")
g.set(ylim=(-0.01, None))
g.set(xticks=[0, 0.0005, 0.001, 0.0015])

In [ ]:
g = lmplot(filter_qcut_chrX.loc[filter_qcut_chrX.Origin.isin(olives_with_tarangire)], x="diversity", y="minor_parent_percentage",
                            scatter=False, hue="Origin", weighted=True, n_boot=1000, ci=99.5) #Additional bootstraps makes it take long
g.set(ylabel="Minor Parent Ancestry Proportion", xlabel="Background Diversity", title="Tanzanian Olive Baboons")
g.set(ylim=(-0.01, None))
g.set(xticks=[0, 0.0005, 0.001, 0.0015])

In [ ]:
g = lmplot(filter_qcut_chrX.loc[filter_qcut_chrX.Origin.isin(gog)], x="diversity", y="minor_parent_percentage",
                            scatter=False, weighted=True, n_boot=1000, ci=99.5) #Additional bootstraps makes it take long
g.set(ylabel="Minor Parent Ancestry Proportion", xlabel="Background Diversity", title="Gog Olive Baboons Chromosome X")
g.set(ylim=(-0.01, None))
g.set(xticks=[0, 0.0005, 0.001, 0.0015])

In [ ]:
g = lmplot(filter_qcut_chrX.loc[filter_qcut_chrX.Origin.isin(["Mahale, Tanzania",
                                                           "Katavi, Tanzania",
                                                           "Issa Valley, Tanzania"])], x="diversity", y="minor_parent_percentage",
                            scatter=False, hue="Origin", weighted=True, n_boot=1000, ci=99.5) #Additional bootstraps makes it take long
g.set(ylabel="Minor Parent Ancestry Proportion", xlabel="Background Diversity", title="Tanzanian Kinda-like Baboons")
g.set(ylim=(-0.01, None))
g.set(xticks=[0, 0.0005, 0.001, 0.0015])

In [ ]:
g = lmplot(filter_qcut_chrX.loc[filter_qcut_chrX.Origin.isin(gog)], x="diversity", y="minor_parent_percentage",
                             hue="Origin", weighted=True, n_boot=1000) #Additional bootstraps makes it take long
g.set(ylabel="Minor Parent Ancestry Proportion", xlabel="Background Diversity", title="Ethiopian Olives")
g.set(ylim=(-0.01, None))
g.set(xticks=[0, 0.0005, 0.001, 0.0015])

In [ ]:
filter_callable_chrX["average_cM_window"].quantile([0.005, 0.995])

In [ ]:
filter_qcut_recomb_chrX = filter_callable_chrX.loc[(filter_callable_chrX.average_cM_window >=1.96e-08) &
                                                      (filter_callable_chrX.average_cM_window <=4.15e-06)]
filter_qcut_recomb_chrX["window_cM"] = filter_qcut_recomb_chrX.average_cM_window*100000

In [ ]:
o_l, slope_l, intercept_l, pval_slope_l, pval_intercept_l, stderr_slope_l, stderr_intercept_l = [], [], [], [], [], [], []
for o in result_order:
    o_div = filter_qcut_recomb_chrX.loc[filter_qcut_recomb_chrX.Origin == o]
    glm_results = smf.glm(formula = "minor_parent_percentage ~ window_cM", data=o_div,
                         var_weights=np.asarray(o_div["diversity"])).fit()
    print(o, glm_results.summary())
    o_l.append(o), slope_l.append(glm_results.params[1]), intercept_l.append(glm_results.params[0])
    pval_slope_l.append(glm_results.pvalues[1]), pval_intercept_l.append(glm_results.pvalues[0])
    stderr_slope_l.append(glm_results.bse[1]), stderr_intercept_l.append(glm_results.bse[0])

In [ ]:
glm_diversity_df = pd.DataFrame({"Origin": o_l, "Intercept": intercept_l, "Slope": slope_l, 
                                 "Intercept P-value": pval_intercept_l, "Slope P-value": pval_slope_l,
                                "Intercept stderr": stderr_intercept_l, "Slope stderr": stderr_slope_l})
glm_diversity_df

In [ ]:
norm_filter_chrX = filter_qcut_chrX.loc[(filter_qcut_chrX.average_cM_window >=1.96e-08) &
                                                      (filter_qcut_chrX.average_cM_window <=4.15e-06)]
norm_filter_chrX["norm_diversity"] = (norm_filter_chrX.diversity-norm_filter_chrX.diversity.mean())/(norm_filter_chrX.diversity.std())
norm_filter_chrX["norm_recomb"] = (norm_filter_chrX.average_cM_window-norm_filter_chrX.average_cM_window.mean())/(norm_filter_chrX.average_cM_window.std())

In [ ]:
autosome_chrX_norm = pd.concat([norm_filter, norm_filter_chrX])
autosome_chrX_norm["chrom_type"] = ["autosome" if x != "chrX" else "chrX" for x in autosome_chrX_norm.chrom]

In [ ]:
o_l, slope_l, pval_slope_l, slope_multi, stderr_slope_l, div_l, pdiv_l = [], [], [], [], [], [], []
for o in result_order:
    o_div = autosome_chrX_norm.loc[(autosome_chrX_norm.Origin == o)]
    glm_results = smf.glm(formula = "minor_parent_percentage ~ norm_diversity * chrom_type", data=o_div,
                         var_weights=np.asarray(o_div["diversity"])).fit()
    print(o, glm_results.summary())
    o_l.append(o), slope_l.append(glm_results.params[3])
    pval_slope_l.append(glm_results.pvalues[3])
    slope_multi.append(glm_results.params[3]/glm_results.params[2])
    div_l.append(glm_results.params[2]), pdiv_l.append(glm_results.pvalues[2])

In [ ]:
glm_diversity_df = pd.DataFrame({"Origin": o_l, "Autosome Slope": div_l, "Interaction Slope": slope_l,
                                 "Relative Increase": slope_multi, "Autosome P-value": pdiv_l,
                                 "Interaction Slope P-value": pval_slope_l
                               })
glm_diversity_df

In [ ]:
selected_pops = ["Mahale, Tanzania","Katavi, Tanzania","Issa Valley, Tanzania"]

o_l, slope_l, pval_slope_l, slope_multi, stderr_slope_l, div_l, pdiv_l = [], [], [], [], [], [], []
for o in selected_pops:
    o_div = autosome_chrX_norm.loc[(autosome_chrX_norm.Origin == o)]
    glm_results = smf.glm(formula = "minor_parent_percentage ~ norm_diversity * chrom_type", data=o_div,
                         var_weights=np.asarray(o_div["diversity"])).fit()
    print(o, glm_results.summary())
    o_l.append(o), slope_l.append(glm_results.params[3])
    pval_slope_l.append(glm_results.pvalues[3])
    slope_multi.append(glm_results.params[3]/glm_results.params[2])
    div_l.append(glm_results.params[2]), pdiv_l.append(glm_results.pvalues[2])

In [ ]:
glm_diversity_df = pd.DataFrame({"Origin": o_l, "Autosome Slope": div_l, "Interaction Slope": slope_l,
                                 "Relative Increase": slope_multi, "Autosome P-value": pdiv_l,
                                 "Interaction Slope P-value": pval_slope_l
                               })
glm_diversity_df

In [ ]:
g = lmplot(autosome_chrX_norm.loc[autosome_chrX_norm.Origin.isin(gog)], x="norm_diversity", y="minor_parent_percentage",
                            scatter=False, hue="chrom_type", n_boot=1000) #Additional bootstraps makes it take long
g.set(ylabel="Minor Parent Ancestry Proportion", xlabel="Normalized Diversity", title="Ethiopian Yellows")
g.set(ylim=(-0.01, None))
g._legend.set(title="Chromosome Type")
#g.set(xticks=[0, 0.0005, 0.001, 0.0015])

In [ ]:
o_l, slope_l, pval_slope_l, slope_multi, stderr_slope_l = [], [], [], [], []
for o in result_order:
    o_div = autosome_chrX_norm.loc[autosome_chrX_norm.Origin == o]
    glm_results = smf.glm(formula = "minor_parent_percentage ~ norm_diversity * chrom_type", data=o_div,
                         var_weights=np.asarray(o_div["diversity"])).fit()
    print(o, glm_results.summary())
    o_l.append(o), slope_l.append(glm_results.params[3])
    pval_slope_l.append(glm_results.pvalues[3]), stderr_slope_l.append(glm_results.bse[3])
    slope_multi.append(glm_results.params[3]/glm_results.params[2])

In [ ]:
o_l, slope_l, pval_slope_l, slope_multi, stderr_slope_l = [], [], [], [], []
for o in range(1):
    o_div = autosome_chrX_norm.loc[autosome_chrX_norm.Origin.isin(["Mahale, Tanzania", "Katavi, Tanzania"])]
    glm_results = smf.glm(formula = "minor_parent_percentage ~ norm_diversity * chrom_type", data=o_div,
                         var_weights=np.asarray(o_div["diversity"])).fit()
    print(o, glm_results.summary())

In [ ]:
g = lmplot(autosome_chrX_norm.loc[autosome_chrX_norm.Origin.isin(gog)], x="diversity", y="minor_parent_percentage",
                            scatter=False, hue="chrom_type", n_boot=1000, ci=99.5) #Additional bootstraps makes it take long
g.set(ylabel="Minor Parent Ancestry Proportion", xlabel="Background Diversity", title="Gog Olive Baboons")
g.set(ylim=(-0.01, None), xlim=(None, 0.0025))
g._legend.set(title="Chromosome Type")

In [ ]:
selected_pops = ["Mahale, Tanzania","Katavi, Tanzania","Issa Valley, Tanzania"]
g = lmplot(autosome_chrX_norm.loc[autosome_chrX_norm.Origin.isin(selected_pops)],
           x="diversity", y="minor_parent_percentage",
            scatter=False, hue="chrom_type", n_boot=1000, col="Origin", col_wrap=3,
          col_order=selected_pops, ci=99.5) #Additional bootstraps makes it take long
g.set(ylabel="Minor Parent Ancestry Proportion", xlabel="Background Diversity")
g.set(ylim=(-0.01, None), xlim=(None, 0.0025))
g.set_titles(col_template="{col_name}")
g._legend.set(title="Chromosome Type", bbox_to_anchor=(1.05, 0.5))

In [ ]:
g = lmplot(autosome_chrX_norm.loc[autosome_chrX_norm.Origin.isin(result_order_no_gog)], x="diversity", y="minor_parent_percentage",
                            scatter=False, hue="chrom_type", n_boot=1000, col="Origin", col_wrap=3, col_order=result_order_no_gog, ci=99.5) #Additional bootstraps makes it take long
g.set(ylabel="Minor Parent Ancestry Proportion", xlabel="Background Diversity")
g.set(ylim=(-0.01, None), xlim=(None, 0.0026))
g.set_titles(col_template="{col_name}")
g._legend.set(title="Chromosome Type")

In [ ]:
glm_diversity_df = pd.DataFrame({"Origin": o_l, "Interaction Slope": slope_l, "Relative Increase": slope_multi, "Slope P-value": pval_slope_l,
                               "Slope stderr": stderr_slope_l})
glm_diversity_df

Removing high freq regions.

In [ ]:
o_l, slope_l, pval_slope_l, slope_multi, stderr_slope_l, div_l, pdiv_l = [], [], [], [], [], [], []
for o in result_order:
    o_div = autosome_chrX_norm.loc[(autosome_chrX_norm.Origin == o) & (autosome_chrX_norm.minor_parent_percentage <= 0.25)]
    glm_results = smf.glm(formula = "minor_parent_percentage ~ norm_diversity * chrom_type", data=o_div,
                         var_weights=np.asarray(o_div["diversity"])).fit()
    print(o, glm_results.summary())
    o_l.append(o), slope_l.append(glm_results.params[3])
    pval_slope_l.append(glm_results.pvalues[3])
    slope_multi.append(glm_results.params[3]/glm_results.params[2])
    div_l.append(glm_results.params[2]), pdiv_l.append(glm_results.pvalues[2])

In [ ]:
glm_diversity_df = pd.DataFrame({"Origin": o_l, "Autosome Slope": div_l, "Interaction Slope": slope_l,
                                 "Relative Increase": slope_multi, "Autosome P-value": pdiv_l,
                                 "Interaction Slope P-value": pval_slope_l
                               })
glm_diversity_df

In [ ]:
g = lmplot(autosome_chrX_norm.loc[autosome_chrX_norm.Origin.isin(result_order_no_gog) &
                                  (autosome_chrX_norm.minor_parent_percentage <= 0.25)], x="diversity", y="minor_parent_percentage",
                            scatter=False, hue="chrom_type", n_boot=1000, col="Origin", col_wrap=3, col_order=result_order_no_gog, ci=99.5) #Additional bootstraps makes it take long
g.set(ylabel="Minor Parent Ancestry Proportion", xlabel="Background Diversity")
g.set(ylim=(-0.01, None), xlim=(None, 0.0026))
g.set_titles(col_template="{col_name}")
g._legend.set(title="Chromosome Type")

Quantiles of Admixture/Diversity

In [ ]:
filter_qcut_chrX["ancestry"] = ["Pure Major" if x <= 0.1 else "Pure Minor" if x >= 0.9
                        else "Mixed" for x in filter_qcut_chrX["minor_parent_percentage"]]
g = sns.FacetGrid(filter_qcut_chrX.loc[filter_qcut_chrX.Origin.isin(result_order)], col="Origin",
                  col_wrap = 4, col_order=result_order)
g.map_dataframe(sns.boxplot, x="diversity", y="ancestry", fliersize=0)
g.set_titles(col_template="{col_name}")
g.set(xlabel="Background Diversity")

In [ ]:
filter_qcut["ancestry"] = ["Pure Major" if x <= 0.1 else "Pure Minor" if x >= 0.9
                        else "Mixed" for x in filter_qcut["minor_parent_percentage"]]
g = sns.FacetGrid(filter_qcut.loc[filter_qcut.Origin.isin(result_order)], col="Origin",
                  col_wrap = 4, col_order=result_order)
g.map_dataframe(sns.boxplot, x="diversity", y="ancestry", fliersize=0)
g.set_titles(col_template="{col_name}")
g.set(xlabel="Background Diversity")

In [ ]:
hama_df = filter_qcut.loc[(filter_qcut.Origin == "Gog Woreda, Ethiopia")][["chrom", "start", "ancestry"]]
hama_df_add = filter_qcut.merge(hama_df, on=["chrom", "start"])
hama_df_chrX = filter_qcut_chrX.loc[(filter_qcut_chrX.Origin == "Gog Woreda, Ethiopia")][["chrom", "start", "ancestry"]]
hama_df_add_chrX = filter_qcut_chrX.merge(hama_df_chrX, on=["chrom", "start"])

In [ ]:
g = sns.FacetGrid(hama_df_add.loc[hama_df_add.Origin.isin(result_order)], col="Origin",
                  col_wrap = 4, col_order=result_order)
g.map_dataframe(sns.boxplot, x="minor_parent_percentage", y="ancestry_y", fliersize=0)
g.set_titles(col_template="{col_name}")
g.set(xlabel="Admixture")

In [ ]:
for o in result_order:
    o_df = hama_df_add.loc[hama_df_add.Origin == o]
    print(o, o_df.groupby(["ancestry_y"])["minor_parent_percentage"].mean(),
         o_df.value_counts(["ancestry_y"]))
    print(mannwhitneyu(o_df.loc[o_df.ancestry_y == "Pure Minor"].minor_parent_percentage,
                   o_df.loc[o_df.ancestry_y == "Mixed"].minor_parent_percentage))

In [ ]:
for o in result_order:
    o_df = hama_df_add_chrX.loc[hama_df_add_chrX.Origin == o]
    print(o, o_df.groupby(["ancestry_y"])["minor_parent_percentage"].mean(),
         o_df.value_counts(["ancestry_y"]))
    print(mannwhitneyu(o_df.loc[o_df.ancestry_y == "Pure Minor"].minor_parent_percentage,
                   o_df.loc[o_df.ancestry_y == "Mixed"].minor_parent_percentage))

Merged quintiles

In [ ]:
filter_qcut_chrX["Diversity Quintile"] = pd.qcut(filter_qcut_chrX.diversity, 5,
                                                       labels=["0-20","20-40",
                                                              "40-60","60-80",
                                                              "80-100"])
filter_qcut_chrX["Chromosome Type"] = "chrX"

filter_qcut["Diversity Quintile"] = pd.qcut(filter_qcut.diversity, 5,
                                                       labels=["0-20","20-40",
                                                              "40-60","60-80",
                                                              "80-100"])
filter_qcut["Chromosome Type"] = "autosome"
filter_qcut_both = pd.concat([filter_qcut, filter_qcut_chrX])
g = sns.FacetGrid(filter_qcut_both.loc[filter_qcut_both.Origin.isin(result_order_no_gog)], col="Origin",
                  col_wrap = 3, col_order=result_order_no_gog, hue="Chromosome Type")
g.map_dataframe(sns.pointplot, y="minor_parent_percentage", x="Diversity Quantile", 
                linestyles="none", capsize=.3, errorbar=("ci", 95), scale = 0.6)
#g.map_dataframe(sns.stripplot, x="minor_parent_percentage", y="Diversity Quantile", alpha=0.05)
#g.set(xlim=(0, 0.25))
g.set_titles(col_template="{col_name}")
g.set(xlabel="Diversity Quintile", ylabel="Minor Parent Ancestry")

In [ ]:
filter_qcut_chrX["Diversity Quintile"] = pd.qcut(filter_qcut_chrX.diversity, 5,
                                                       labels=["0-20","20-40",
                                                              "40-60","60-80",
                                                              "80-100"])
filter_qcut_chrX["Chromosome Type"] = "chrX"

filter_qcut["Diversity Quintile"] = pd.qcut(filter_qcut.diversity, 5,
                                                       labels=["0-20","20-40",
                                                              "40-60","60-80",
                                                              "80-100"])
filter_qcut["Chromosome Type"] = "autosome"
filter_qcut_both = pd.concat([filter_qcut, filter_qcut_chrX])
g = sns.FacetGrid(filter_qcut_both.loc[filter_qcut_both.Origin.isin(["Mahale, Tanzania", "Katavi, Tanzania",
                                                                     "Issa Valley, Tanzania"])], col="Origin",
                  col_wrap = 3, col_order=result_order_no_gog, hue="Chromosome Type")
g.map_dataframe(sns.pointplot, y="minor_parent_percentage", x="Diversity Quantile", 
                linestyles="none", capsize=.3, errorbar=("ci", 95), scale = 0.6)
#g.map_dataframe(sns.stripplot, x="minor_parent_percentage", y="Diversity Quantile", alpha=0.05)
#g.set(xlim=(0, 0.25))
g.set_titles(col_template="{col_name}")
g.set(xlabel="Diversity Quintile", ylabel="Minor Parent Ancestry")

In [ ]:
mean_window_df_tanz_eth = window_df_tanz_eth.groupby(["chrom", "Origin", "individual", "start", "end"])[["north_sum"]].mean().reset_index()
mean_window_df_tanz = window_df_tanz.groupby(["chrom", "Origin", "individual", "start", "end"])[["north_sum"]].mean().reset_index()
admix_div_mean_ind = mean_diversity.merge(mean_window_df_tanz_eth, on=["chrom", "start"])
admix_div_mean_ind["Species"] = admix_div_mean_ind.Origin.map(dict(zip(meta_data_samples_sci.Origin, meta_data_samples_sci.Species)))
admix_div_mean_ind = admix_div_mean_ind.merge(c_r_g_df, on=["chrom", "start"])
admix_div_mean_ind["North Percentage"] = admix_div_mean_ind.north_sum/100000

In [ ]:
admix_div_mean_ind = admix_div_mean_ind.loc[admix_div_mean_ind.Origin.isin(origins_interest)]
admix_div_mean_ind["minor_parent_percentage"] = [x if y == ("cynocephalus") or z == "Gog Woreda, Gambella region, Ethiopia_eth_case"
                                             else 1-x for x, y, z in zip(admix_div_mean_ind["North Percentage"],
                                                                         admix_div_mean_ind["Species"],
                                                                        admix_div_mean_ind["Origin"])]
admix_div_mean_ind["local_minor_ancestry"] = [min(x, 1-x) for x in admix_div_mean_ind["North Percentage"]]

In [ ]:
filter_callable_ind = admix_div_mean_ind.loc[admix_div_mean_ind.callable_frac > 0.75]
filter_qcut_ind = filter_callable_ind.loc[(filter_callable_ind.diversity >= 0.000513) &
                                 (filter_callable_ind.diversity <= 0.00408)]

In [ ]:
o_l, id_l, slope_l, intercept_l, pval_slope_l, pval_intercept_l, stderr_slope_l, stderr_intercept_l = [], [], [], [], [], [], [], []
for o in result_order:
    o_div = filter_qcut_ind.loc[(filter_qcut_ind.Origin == o)]
    for i in o_div.individual.unique():
        i_df = o_div.loc[o_div.individual == i]
        glm_results = smf.glm(formula = "minor_parent_percentage ~ diversity", data=i_df,
                         var_weights=np.asarray(i_df["diversity"])).fit()
        print(o, i, glm_results.summary())
        o_l.append(o), id_l.append(i)
        slope_l.append(glm_results.params[1]), intercept_l.append(glm_results.params[0])
        pval_slope_l.append(glm_results.pvalues[1]), pval_intercept_l.append(glm_results.pvalues[0])
        stderr_slope_l.append(glm_results.bse[1]), stderr_intercept_l.append(glm_results.bse[0])

In [ ]:
glm_diversity_df = pd.DataFrame({"Origin": o_l, "individual": id_l, "Intercept": intercept_l, "Slope": slope_l, 
                                 "Intercept P-value": pval_intercept_l, "Slope P-value": pval_slope_l,
                                "Intercept stderr": stderr_intercept_l, "Slope stderr": stderr_slope_l})
glm_diversity_df

In [ ]:
sns.histplot(data=glm_diversity_df.loc[glm_diversity_df.Origin == "Serengeti, Tanzania"], x="Slope")

In [ ]:
glm_diversity_df.loc[glm_diversity_df.Origin == "Serengeti, Tanzania"]

In [ ]:
g = lmplot(filter_qcut_ind.loc[filter_qcut_ind.Origin.isin(["Ruaha, Tanzania"])], x="diversity", y="minor_parent_percentage",
                            scatter=False, hue="individual", ci=99.5) #Additional bootstraps makes it take long
g.set(ylabel="Minor Parent Ancestry Proportion", xlabel="Background Diversity", title="Tanzanian Yellow Baboons")
g.set(ylim=(-0.01, None))
g.set(xticks=np.arange(0.0005, 0.004, 0.001))

In [ ]:
g = lmplot(admix_div_mean_ind.loc[admix_div_mean_ind.Origin.isin(yellows)], x="diversity", y="minor_parent_percentage",
                            scatter=False, hue="Origin", weighted=True, n_boot=1000, ci=99.5) #Additional bootstraps makes it take long
g.set(ylabel="Minor Parent Ancestry Proportion", xlabel="Background Diversity", title="Tanzanian Yellow Baboons")
g.set(ylim=(-0.01, None))
g.set(xticks=np.arange(0.0005, 0.004, 0.001))

Admixture per chromosome investigation

In [ ]:
ind_df = window_df_tanz_eth.groupby(["individual", "chrom", "Origin", "start", "end"])[["north_sum"]].mean().reset_index()
ind_df_div = mean_diversity.merge(ind_df, on=["chrom", "start"])
ind_df_div = ind_df_div.merge(c_r_g_df, on=["chrom", "start"])
ind_df_div["North Percentage"] = ind_df_div.north_sum/100000
ind_df_div["Species"] = ind_df_div.Origin.map(dict(zip(meta_data_samples_sci.Origin, meta_data_samples_sci.Species)))

#Selecting the cases of interest and setting Minor Parent Ancestry

ind_df_div = ind_df_div.loc[ind_df_div.Origin.isin(origins_interest)]
ind_df_div["minor_parent_percentage"] = [x if y == ("cynocephalus") or z == "Gog Woreda, Gambella region, Ethiopia_eth_case"
                                             else 1-x for x, y, z in zip(ind_df_div["North Percentage"],
                                                                         ind_df_div["Species"],
                                                                        ind_df_div["Origin"])]
ind_df_div = ind_df_div.loc[ind_df_div.callable_frac > 0.75]

In [ ]:
o_l, low_l, high_l, effect_l, admix_l, chr_l, ind_l = [], [], [], [], [], [], []
#filter_qcut_chrX["Diversity Quantile"] = pd.qcut(filter_qcut_chrX.diversity, 5,
#                                                       labels=["0-20","20-40",
#                                                              "40-60","60-80",
#                                                              "80-100"])

for o in ind_df_div.Origin.unique():
    print(o)
    for c in ind_df_div.chrom.unique():
        print(c)
        for i in ind_df_div.loc[(ind_df_div.Origin == o) & (ind_df_div.chrom == c)].individual.unique():
            i_s_df = ind_df_div.loc[(ind_df_div.individual == i) & (ind_df_div.chrom == c)]
            #print(i)
            o_l.append(o), ind_l.append(i), chr_l.append(c)
            i_s_df["Diversity Quantile"] = pd.qcut(i_s_df.diversity, 2,
                                                       labels=["0-50", "50-100"])
            low_mean = i_s_df.loc[i_s_df["Diversity Quantile"] == "0-50"][["minor_parent_percentage"]].mean()[0]*100
            high_mean = i_s_df.loc[i_s_df["Diversity Quantile"] == "50-100"][["minor_parent_percentage"]].mean()[0]*100
            low_l.append(low_mean)
            high_l.append(high_mean)
            effect_l.append(high_mean/low_mean)
            admix_l.append(i_s_df.minor_parent_percentage.mean())

In [ ]:
admix_df = pd.DataFrame({"Origin": o_l, "individual": ind_l, "chrom": chr_l, "Low Diversity Quantile": low_l,
              "High Diversity Quantile": high_l, "Relative Difference": effect_l, "Mean_Admixture": admix_l})
admix_df["Absolute_Difference"] = admix_df["High Diversity Quantile"]-admix_df["Low Diversity Quantile"]
admix_df["Concentration"] = admix_df["High Diversity Quantile"]/admix_df.Mean_Admixture*0.5

In [ ]:
sns.lmplot(admix_df.loc[admix_df.Origin == "Ruaha, Tanzania"],
           x="Mean_Admixture", y="Concentration")

In [ ]:
admix_df["Mean Admixture Bin"] = pd.cut(admix_df.Mean_Admixture, [0, 0.02, 0.04, 0.06,
                                                                  0.08, 0.1, 0.2, 1])
g = sns.FacetGrid(admix_df.loc[admix_df.Origin.isin(result_order_no_gog)], col="Origin",
                  col_wrap = 3, col_order=result_order_no_gog)
g.map_dataframe(sns.stripplot, y="Mean Admixture Bin", x="Concentration", alpha=.1, color="grey")
g.map_dataframe(sns.pointplot, y="Mean Admixture Bin", x="Concentration", 
                linestyles="none", capsize=.3, errorbar=("ci", 95), scale = 0.6)
#g.map_dataframe(sns.stripplot, x="minor_parent_percentage", y="Diversity Quantile", alpha=0.05)
g.set(xlim=(25, 75))
g.set_titles(col_template="{col_name}")
axes = g.axes.flatten()
for ax in g.axes.flat:
    ax.axvline(x=50, color='r', linestyle=':')
g.set(xlabel="MPA Concentration", ylabel="Autosome Admixture Bin")

In [ ]:
admix_df["Mean Admixture Bin"] = pd.cut(admix_df.Mean_Admixture, [0, 0.02, 0.04, 0.06,
                                                                  0.08, 0.1, 1])
admix_df["Species"] = admix_df.Origin.map(dict(zip(meta_data_samples_sci.Origin, meta_data_samples_sci.Species)))
g = sns.FacetGrid(admix_df.loc[admix_df.Origin.isin(result_order_no_gog)], col="Species",
                  col_wrap = 2)
g.map_dataframe(sns.pointplot, y="Mean Admixture Bin", x="Concentration", 
                linestyles="none", capsize=.3, errorbar=("ci", 95), scale = 0.6)
#g.map_dataframe(sns.stripplot, x="minor_parent_percentage", y="Diversity Quantile", alpha=0.05)
#g.set(xlim=(0, 0.25))
g.set_titles(col_template="{col_name}")
g.set(xlabel="Absolute Difference", ylabel="Mean Minor Parent Ancestry")

In [ ]:
glm_results = smf.glm(formula = "Absolute_Difference ~ Mean_Admixture+I(Mean_Admixture**2)",
                      data=admix_df.loc[admix_df.Origin == "Ruaha, Tanzania"]).fit()
print(glm_results.summary())

In [ ]:
glm_results = smf.glm(formula = "Absolute_Difference ~ Mean_Admixture", data=admix_df).fit()
print(glm_results.summary())